# Spectral Analysis of Tropical Variability: From Fourier Transform to MJO Time Scales

---

This notebook introduces spectral analysis using daily tropical climate data.

We will start with the basic idea of Fourier transform, then move toward spectral analysis and its application to the Madden–Julian Oscillation (MJO).

The main variables used in this notebook are:

- OLR: outgoing longwave radiation, used as a proxy for tropical deep convection
- U850: 850-hPa zonal wind, used to describe lower-tropospheric circulation
- U200: 200-hPa zonal wind, used to describe upper-tropospheric circulation
- SST: sea surface temperature, used as an optional variable for air-sea interaction

The data period is:

$$
1979\text{-}01\text{-}01 \ \text{to} \ 2024\text{-}12\text{-}31
$$

The main scientific focus is tropical variability on subseasonal time scales, especially the 30–90 day time scale associated with MJO-like variability.

## Learning goals

By the end of this notebook, you should be able to:

1. Explain the difference between the time domain and the frequency domain.

2. Use Fourier transform to decompose a time series into different frequencies or periods.

3. Interpret a power spectrum as a way to describe how variance is distributed across time scales.

4. Distinguish between a broad-spectrum view and an MJO-focused view.

5. Understand why preprocessing choices matter for real climate data.

6. Explain why red noise is a useful null hypothesis for climate spectra.

7. Use OLR to identify subseasonal tropical convective variability.

8. Use OLR, U850, and U200 together to understand MJO-related convection-circulation coupling.

9. Use lead-lag correlation to examine phase relationships among MJO-related variables.

## Scientific motivation: why spectral analysis for MJO?

Daily climate data contain variability on many different time scales.

For example, a daily OLR time series may include weather-scale variability, the seasonal cycle, subseasonal variability, interannual variability, and longer-term variability.

A simple way to think about this is:

$$
x(t)
=
\bar{x}
+
x_{\mathrm{seasonal}}(t)
+
x_{\mathrm{subseasonal}}(t)
+
x_{\mathrm{interannual}}(t)
+
x_{\mathrm{low-frequency}}(t)
+
noise(t)
$$

Here:

- $\bar{x}$ is the time mean.
- $x_{\mathrm{seasonal}}(t)$ is the seasonal cycle.
- $x_{\mathrm{subseasonal}}(t)$ includes variability shorter than a season but longer than weather noise.
- $x_{\mathrm{interannual}}(t)$ includes year-to-year variability, such as ENSO-related variability.
- $x_{\mathrm{low-frequency}}(t)$ includes decadal variability or long-term changes.
- $noise(t)$ represents irregular high-frequency fluctuations.

The Madden-Julian Oscillation, or MJO, is usually associated with tropical variability on approximately 30–90 day time scales:

$$
30 \ \text{days} \leq P \leq 90 \ \text{days}
$$

where $P$ is the period of variability.

However, MJO is not defined only by its time scale. It also has spatial structure, eastward propagation, and coupling between tropical convection and large-scale circulation.

Therefore, this notebook starts with spectra of time series, then later connects the spectra to MJO-related physical interpretation.

## Variables used in this notebook

We will mainly use three atmospheric variables.

### OLR

OLR stands for outgoing longwave radiation.

In the tropics, OLR is often used as a proxy for deep convection.

Lower OLR usually indicates colder cloud tops and stronger deep convection.

Higher OLR usually indicates suppressed convection or clearer-sky conditions.

### U850

U850 is the zonal wind at 850 hPa.

It represents lower-tropospheric zonal wind.

For MJO analysis, U850 helps describe the low-level circulation associated with tropical convection.

### U200

U200 is the zonal wind at 200 hPa.

It represents upper-tropospheric zonal wind.

Together, U850 and U200 help describe the vertical structure of MJO-related circulation anomalies.

### SST

SST stands for sea surface temperature.

SST is not the primary variable for identifying MJO, but it can be useful for studying air-sea interaction associated with MJO variability.

In this notebook, SST can be treated as an optional extension.

## Broad-spectrum view versus MJO-focused view

There are two related but different goals in this notebook.

### Goal 1: broad-spectrum view

First, we may want to see what time scales are present in the original data.

For this goal, we usually remove only the time mean:

$$
x'(t) = x(t) - \bar{x}
$$

This keeps the seasonal cycle, subseasonal variability, interannual variability, and lower-frequency variability.

This is useful if we want to see multiple time scales in one spectrum.

### Goal 2: MJO-focused view

Later, if we want to focus specifically on MJO-like subseasonal variability, we remove the mean seasonal cycle:

$$
x'(t) = x(t) - \overline{x}_{doy}
$$

where $\overline{x}_{doy}$ is the day-of-year climatology.

This removes the regular seasonal cycle and makes 30–90 day variability easier to examine.

The key point is:

> Anomaly does not always mean the same thing.

A demeaned anomaly removes only the time mean.

A seasonal-cycle-removed anomaly removes the mean seasonal cycle.

The correct choice depends on the scientific question.

## Import packages

We first import the Python packages needed for this notebook.

The main packages are:

- `s3fs` for accessing Zarr datasets stored on cloud object storage. It allows `xarray` to open remote datasets without downloading the full files locally.
- `numpy` for numerical calculations
- `xarray` for working with gridded climate data
- `pandas` for time handling
- `matplotlib` for plotting
- `scipy.signal` for spectral analysis and filtering
- `scipy.stats` for basic statistical calculations

In [ ]:
import s3fs
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

from scipy import signal
from scipy import stats

## User settings

Here we define basic settings for the analysis.

The data period is from 1979-01-01 to 2024-12-31.

The default tropical averaging region is:

$$
10^\circ S - 10^\circ N,\quad 60^\circ E - 160^\circ E
$$

This region covers much of the tropical Indian Ocean and western Pacific, where MJO-related convection is often active.

We also define:

$$
\Delta t = 1 \ \text{day}
$$

because the data are daily.

In [ ]:
# -------------------------------
# Time period
# -------------------------------

start_date = "1979-01-01"
end_date = "2024-12-31"

# -------------------------------
# Default tropical analysis region
# -------------------------------

lat_bounds = (-10, 10)
lon_bounds = (60, 160)

# -------------------------------
# Sampling interval
# -------------------------------

# Daily data: one data point per day
dt = 1.0  # unit: day

## File paths and variable names

In this notebook, you should replace the file paths below with the paths to your own data files.

The variable names may also differ depending on your dataset.

Common examples are:

- OLR variable name: `olr`
- U850 variable name: `u850`
- U200 variable name: `u200`
- SST variable name: `sst`

Check your dataset carefully before running the analysis.

In [ ]:
# -------------------------------
# File paths
# Replace these with your own files
# -------------------------------
URL = 'https://js2.jetstream-cloud.org:8001/' #Locate and read a file
fs = s3fs.S3FileSystem(anon=True, client_kwargs=dict(endpoint_url=URL))

# OLR
olr_noaa_store = s3fs.S3Map(
    root=f'pythia/olr_noaa.zarr',
    s3=fs,
    check=False
)
olr = xr.open_zarr(olr_noaa_store)
olr = olr.rename_vars({'__xarray_dataarray_variable__': 'olr'})

# SST
sst_noaa_oi_store = s3fs.S3Map(
    root=f'pythia/sst_noaa_oi.zarr',
    s3=fs,
    check=False
)
sst = xr.open_zarr(sst_noaa_oi_store)
sst = sst.rename_vars({'__xarray_dataarray_variable__': 'sst'})
# Zonal wind
uwind_ncep_ncar_store = s3fs.S3Map(
    root=f'pythia/uwind-ncep-ncar.zarr',
    s3=fs,
    check=False
)
# Open with xarray
uwind_ncep_ncar = xr.open_zarr(uwind_ncep_ncar_store)
u850 = uwind_ncep_ncar.isel(level=0).rename_vars({'uwnd': 'u850'})
u200 = uwind_ncep_ncar.isel(level=1).rename_vars({'uwnd': 'u200'})

## A helper function for area averaging

Many parts of this notebook use an area-mean time series.

For example, we may want the average OLR over:

$$
10^\circ S - 10^\circ N,\quad 60^\circ E - 160^\circ E
$$

Because longitude-latitude grid boxes have different physical areas at different latitudes, we use cosine-latitude weighting.

The weight is:

$$
w(\phi) = \cos(\phi)
$$

where $\phi$ is latitude.

This gives less weight to high-latitude grid cells and more appropriate area weighting.

In [ ]:
def area_mean(da, lat_bounds, lon_bounds, lat_name="lat", lon_name="lon"):
    """
    Compute cosine-latitude-weighted area mean.

    Parameters
    ----------
    da : xarray.DataArray
        Input data with latitude and longitude dimensions.
    lat_bounds : tuple
        Latitude range, for example (-10, 10).
    lon_bounds : tuple
        Longitude range in 0-360 convention, for example (60, 160).
    lat_name : str
        Name of latitude coordinate.
    lon_name : str
        Name of longitude coordinate.

    Returns
    -------
    da_mean : xarray.DataArray
        Area-mean time series.
    """

    lat1, lat2 = lat_bounds
    lon1, lon2 = lon_bounds

    # Select the region
    da_reg = da.sel(
        {
            lat_name: slice(lat1, lat2),
            lon_name: slice(lon1, lon2),
        }
    )

    # Cosine-latitude weights
    weights = np.cos(np.deg2rad(da_reg[lat_name]))

    # Weighted spatial mean
    da_mean = da_reg.weighted(weights).mean(dim=(lat_name, lon_name))

    return da_mean

## Notebook structure

### Part I. Fourier transform intuition

We begin with a synthetic time series that contains known 30-day, 60-day, and 365-day oscillations.

This part explains the basic math behind Fourier transform.

### Part II. Statistical spectral analysis

We then discuss why raw spectra can be noisy, why red noise matters, and how confidence levels help us interpret spectral peaks.

### Part III. Real OLR spectrum

We apply the method to real daily OLR from 1979 to 2024.

We first examine a broad-spectrum view, then move to MJO-focused OLR anomalies.

### Part IV. MJO convection-circulation coupling

We add U850 and U200 to examine whether subseasonal OLR variability is connected to lower- and upper-tropospheric circulation.

### Part V. MJO propagation

We use longitude-time diagrams to examine whether filtered OLR anomalies show eastward propagation.

### Optional extensions

If time allows, we can also examine SST and air-sea interaction, or introduce wavenumber-frequency spectra.

# Part I. Fourier transform intuition with synthetic data

Before applying spectral analysis to real OLR, SST, and wind data, we first use a synthetic time series.

The advantage of a synthetic example is that we already know the correct answer.

We will build a time series that contains:

- a 30-day oscillation
- a 60-day oscillation
- a 365-day annual cycle
- random noise

Then we will use Fourier transform to recover these known periods.

This part focuses on the basic idea:

$$
x(t) \rightarrow X(f)
$$

That means we convert a signal from the time domain to the frequency domain.

## 1. Time domain versus frequency domain

A climate time series is usually first viewed in the **time domain**.

In the time domain, we ask:

> How does the variable change with time?

Mathematically, we write the signal as:

$$
x(t)
$$

where $x$ is the variable and $t$ is time.

For example, $x(t)$ could be daily OLR averaged over a tropical region.

However, for MJO analysis, we often want to ask a different question:

> Which time scales are present in the signal?

To answer this, we move from the time domain to the **frequency domain**.

In the frequency domain, we describe the signal as:

$$
X(f)
$$

where $f$ is frequency.

Fourier transform is the mathematical tool that converts:

$$
x(t) \rightarrow X(f)
$$

In simple terms, Fourier transform helps us identify which periods or frequencies make up a time series.

## 2. A simple wave

The basic building block of Fourier analysis is a simple wave.

A cosine wave can be written as:

$$
x(t) = A \cos(\omega t - \phi)
$$

Here:

- $x(t)$ is the value of the signal at time $t$.
- $A$ is the amplitude.
- $\omega$ is the angular frequency.
- $\phi$ is the phase.

The amplitude $A$ tells us how large the oscillation is.

The angular frequency $\omega$ tells us how fast the wave oscillates.

The phase $\phi$ tells us where the wave is shifted in time.

In a plot, these parameters affect the wave in different ways:

- $A$ changes the height of the wave.
- $\omega$ changes how closely spaced the peaks are.
- $\phi$ shifts the wave left or right in time.

## 3. Frequency and period

In climate science, we often think in terms of **period** rather than frequency.

The frequency $f$ tells us how many cycles occur per unit time.

The period $P$ tells us how long one complete cycle takes.

They are related by:

$$
P = \frac{1}{f}
$$

For daily data, frequency has units of cycles per day.

For example, if:

$$
f = \frac{1}{30} \ \text{cycles per day}
$$

then:

$$
P = 30 \ \text{days}
$$

So a 30-day oscillation has frequency $1/30$ cycles per day.

Angular frequency is related to frequency by:

$$
\omega = 2\pi f
$$

Since $f = 1/P$, we can also write:

$$
\omega = \frac{2\pi}{P}
$$

For a 30-day oscillation:

$$
\omega = \frac{2\pi}{30}
$$

For a 365-day annual cycle:

$$
\omega = \frac{2\pi}{365}
$$

The 30-day oscillation has a larger angular frequency than the 365-day oscillation because it repeats more quickly.

## 4. Create a synthetic daily time axis

We first create a daily time axis.

This synthetic example lasts for 10 years.

Because we are using daily data, the sampling interval is:

$$
\Delta t = 1 \ \text{day}
$$

In [ ]:
# -------------------------------
# Create a synthetic daily time axis
# -------------------------------

# Number of years
nyears = 10

# Daily sampling: one value per day
dt = 1.0  # unit: day

# Total number of time steps
nt = int(nyears * 365)

# Time array in days
t = np.arange(nt) * dt

## 5. Create known oscillations

Now we create a synthetic time series with known periods.

The signal contains three periodic components:

$$
x_{30}(t)
$$

$$
x_{60}(t)
$$

$$
x_{365}(t)
$$

These represent 30-day, 60-day, and 365-day oscillations.

The full synthetic signal is:

$$
x(t) = x_{30}(t) + x_{60}(t) + x_{365}(t) + noise(t)
$$

This is useful for teaching because we know exactly which periods we put into the signal.

If Fourier transform works, the spectrum should show strong power near 30 days, 60 days, and 365 days.

In [ ]:
# -------------------------------
# Create known oscillations
# -------------------------------

# 30-day oscillation
period_30 = 30.0
signal_30 = 2.0 * np.cos(2 * np.pi * t / period_30)

# 60-day oscillation
period_60 = 60.0
signal_60 = 1.5 * np.cos(2 * np.pi * t / period_60)

# 365-day seasonal cycle
period_365 = 365.0
signal_365 = 4.0 * np.cos(2 * np.pi * t / period_365)

# Random noise
np.random.seed(42)
noise = np.random.normal(loc=0.0, scale=2.0, size=nt)

# Final synthetic time series
x = signal_30 + signal_60 + signal_365 + noise

## 6. Plot the synthetic time series

We first plot the signal in the time domain.

This plot shows how the synthetic signal changes with time.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(t, x, linewidth=1)

ax.set_xlabel("Time [days]")
ax.set_ylabel("Synthetic signal")
ax.set_title("Synthetic daily time series")

plt.show()

The annual cycle is the most visible large-scale feature in this synthetic time series.

Because the synthetic record is 10 years long, we can see about 10 large annual cycles.

However, the 30-day and 60-day oscillations are harder to identify by eye because they are mixed with each other and with random noise.

This illustrates a key motivation for Fourier transform:

> A time-domain plot may show that variability exists, but it does not easily quantify which periods explain the variability.

In other words, the time-domain plot is useful, but it is not enough if we want to objectively identify different time scales.

## 7. Look at the individual components

Before computing the Fourier transform, it is helpful to plot the components separately.

This allows us to see what each part contributes to the full signal.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(t, signal_365, label="365-day annual cycle", linewidth=2)
ax.plot(t, signal_60, label="60-day oscillation", linewidth=1)
ax.plot(t, signal_30, label="30-day oscillation", linewidth=1)

ax.set_xlabel("Time [days]")
ax.set_ylabel("Signal")
ax.set_title("Individual periodic components")
ax.legend()

plt.show()

The 365-day annual cycle is a low-frequency component because it changes slowly.

The 30-day and 60-day oscillations are higher-frequency components because they repeat more often.

In general:

- long period means low frequency
- short period means high frequency

The Fourier transform separates these components by frequency or period.

Now we compare the full synthetic signal with the annual cycle alone.

This helps us see that the annual cycle is present in the full signal, even though shorter-period oscillations and random noise are also present.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(t, x, label="Full synthetic signal", linewidth=0.8)
ax.plot(t, signal_365, label="365-day annual cycle only", linewidth=2)

ax.set_xlabel("Time [days]")
ax.set_ylabel("Signal")
ax.set_title("Full signal compared with the annual cycle")
ax.legend()

plt.show()

The annual cycle is visible as the large-scale repeating pattern.

The shorter-period oscillations and noise create rapid fluctuations around this large-scale pattern.

This is similar to real climate data: one observed time series can contain multiple time scales at the same time.

We can also zoom in on the first two years.

A shorter time window makes the 30-day and 60-day oscillations easier to see.

In [ ]:
# Select the first two years
nplot = 2 * 365

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(t[:nplot], x[:nplot], label="Full synthetic signal", linewidth=0.9)
ax.plot(t[:nplot], signal_365[:nplot], label="365-day annual cycle", linewidth=2)

ax.set_xlabel("Time [days]")
ax.set_ylabel("Signal")
ax.set_title("First two years of the synthetic signal")
ax.legend()

plt.show()

This zoomed-in plot shows an important practical point:

The time scale that appears most obvious can depend on the time range we plot.

A long time range makes slow variability easier to see.

A short time range makes faster variability easier to see.

A spectrum gives a more systematic way to compare variability across many time scales.

## 8. Compute the Fourier transform

Now we compute the Fourier transform of the synthetic signal.

The goal is to convert the signal from the time domain to the frequency domain.

Before applying the Fourier transform, we remove the time mean:

$$
x'(t) = x(t) - \bar{x}
$$

The mean value corresponds to zero frequency.

Since we are interested in variability around the mean, we remove it first.

In [ ]:
# -------------------------------
# Step 1: Remove the time mean
# -------------------------------

# The mean value corresponds to zero frequency.
# We remove it because we are interested in variability around the mean.
x_anom = x - np.mean(x)

# -------------------------------
# Step 2: Compute Fourier transform
# -------------------------------

# Fourier coefficients
fft_coeff = np.fft.rfft(x_anom)

# Corresponding frequencies
freq = np.fft.rfftfreq(nt, d=dt)

The output `fft_coeff` contains the Fourier coefficients.

These coefficients are complex numbers.

A complex Fourier coefficient contains information about both:

- amplitude: how strong this frequency component is
- phase: where this wave is located in time

In this first part of the notebook, we focus on amplitude and power.

The array `freq` contains the frequencies associated with the Fourier coefficients.

Because we use daily data and set:

$$
\Delta t = 1 \ \text{day}
$$

the frequency unit is cycles per day.

## 9. Convert frequency to period

For climate interpretation, period is often easier to understand than frequency.

The relationship is:

$$
P = \frac{1}{f}
$$

The Fourier transform includes a zero-frequency component.

This component represents the time mean.

Because the period would be infinite when $f=0$, we remove the zero-frequency component before converting frequency to period.

In [ ]:
# Avoid dividing by zero frequency
freq_nonzero = freq[1:]
fft_nonzero = fft_coeff[1:]

# Convert frequency to period
period = 1 / freq_nonzero

Now `period` contains the period corresponding to each Fourier frequency.

Because the data are daily, the period is measured in days.

For example:

- period near 30 means a 30-day oscillation
- period near 60 means a 60-day oscillation
- period near 365 means an annual cycle

## 10. Compute amplitude and power

The Fourier coefficient at each frequency is written as:

$$
X(f)
$$

The amplitude is the absolute value of the Fourier coefficient:

$$
A(f) = |X(f)|
$$

The power is proportional to the square of the amplitude:

$$
Power(f) = |X(f)|^2
$$

This is important because variance is related to amplitude squared.

A wave with twice the amplitude contributes about four times the power.

So the power spectrum helps us understand how much variability is associated with each frequency or period.

In [ ]:
# -------------------------------
# Amplitude spectrum
# -------------------------------

amplitude = np.abs(fft_nonzero)

# -------------------------------
# Power spectrum
# -------------------------------

power = amplitude ** 2

At this stage, `power` is the raw Fourier power.

It is useful for identifying spectral peaks, but its absolute value depends on the length of the time series and the normalization convention.

For teaching purposes, we first plot the raw power spectrum.

Later, we will normalize the power so that it can be interpreted as variance explained.

## 11. Plot the raw power spectrum

Now we plot the raw power as a function of period.

The x-axis is period in days.

We use a logarithmic x-axis because climate variability covers a wide range of time scales, from a few days to hundreds of days.

We also use a logarithmic y-axis because spectral power can vary by several orders of magnitude.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(period, power, linewidth=1)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel("Period [days]")
ax.set_ylabel("Raw power")
ax.set_title("Raw Fourier power spectrum of the synthetic time series")

# Make the x-axis easier to read
ax.set_xlim(2, 1000)
ax.invert_xaxis()

plt.show()

In this figure, large periods are on the left and short periods are on the right.

This layout is often useful in climate spectral analysis because it separates slow variability from fast variability.

In our synthetic example, the time series was constructed using three known periodic components:

- 365-day annual cycle
- 60-day oscillation
- 30-day oscillation

So the spectrum should show enhanced power near these periods.

This is the key advantage of Fourier transform:

> Even if multiple oscillations are mixed together in the time-domain plot, the spectrum can separate them by period.

## 12. A cleaner function for Fourier power spectrum

Now we write a reusable function.

This function takes a one-dimensional time series and returns:

- frequency
- period
- Fourier amplitude
- raw power
- normalized power
- percent variance

The function assumes that:

1. The data are evenly spaced in time.
2. The data have no missing values.
3. The input is one-dimensional.

In [ ]:
def compute_fft_spectrum(x, dt=1.0):
    """
    Compute a simple one-sided Fourier power spectrum.

    Parameters
    ----------
    x : array-like
        One-dimensional time series.
    dt : float
        Sampling interval.
        For daily data, dt = 1 day.

    Returns
    -------
    spectrum : dict
        Dictionary containing frequency, period, amplitude, raw power,
        normalized power, percent variance, and total variance.
    """

    # Convert input to a NumPy array
    x = np.asarray(x)

    # Check for missing values
    if np.any(np.isnan(x)):
        raise ValueError(
            "Input time series contains NaN values. "
            "Please remove or fill missing values before applying FFT."
        )

    # Number of time steps
    n = len(x)

    # Remove the time mean
    x_anom = x - np.mean(x)

    # Variance in the time domain
    var_x = np.var(x_anom)

    # Compute real FFT
    fft_coeff = np.fft.rfft(x_anom)

    # Compute corresponding frequencies
    freq = np.fft.rfftfreq(n, d=dt)

    # Remove zero frequency
    freq = freq[1:]
    fft_coeff = fft_coeff[1:]

    # Convert frequency to period
    period = 1 / freq

    # Amplitude
    amplitude = np.abs(fft_coeff)

    # Raw power
    raw_power = amplitude ** 2

    # Normalize power so that it is comparable to variance.
    # For a one-sided spectrum, multiply by 2 because the negative-frequency
    # part of the spectrum is not returned by rfft.
    normalized_power = (2 * raw_power) / (n ** 2)

    # Convert normalized power to percent variance
    percent_variance = 100 * normalized_power / var_x

    spectrum = {
        "frequency": freq,
        "period": period,
        "amplitude": amplitude,
        "raw_power": raw_power,
        "normalized_power": normalized_power,
        "percent_variance": percent_variance,
        "variance": var_x,
    }

    return spectrum

The normalization step makes the spectrum easier to interpret statistically.

The normalized power tells us how much variance is associated with each Fourier frequency.

The percent variance is calculated as:

$$
\text{Percent variance}(f)
=
100
\times
\frac{\text{Normalized power}(f)}
{\text{Total variance}}
$$

This means the y-axis can be interpreted as the percentage of total variance associated with each frequency.

For this introductory example, the exact normalization is less important than the main idea:

> Larger spectral power means that a period contributes more strongly to the variability of the time series.

## 13. Apply the function to the synthetic signal

Now we apply the function to the synthetic time series.

Because the data are daily, we use:

$$
dt = 1.0
$$

This means the sampling interval is one day.

In [ ]:
spec = compute_fft_spectrum(x, dt=1.0)

period = spec["period"]
percent_variance = spec["percent_variance"]

Now we plot percent variance as a function of period.

This version is easier to interpret than the raw power spectrum because the y-axis tells us how much variance is explained by each period.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(period, percent_variance, linewidth=1)

ax.set_xscale("log")

ax.set_xlabel("Period [days]")
ax.set_ylabel("Variance explained [%]")
ax.set_title("Fourier spectrum of the synthetic signal")

ax.set_xlim(2, 1000)
ax.invert_xaxis()

# Mark the known periods used to create the synthetic signal
for p in [30, 60, 365]:
    ax.axvline(p, linestyle="--", linewidth=1)
    ax.text(
        p,
        ax.get_ylim()[1] * 0.7,
        f"{p} d",
        rotation=90,
        va="center"
    )

plt.show()

The dashed vertical lines mark the periods that we used to construct the synthetic signal.

We should see enhanced variance near:

- 365 days
- 60 days
- 30 days

This confirms that the Fourier transform can recover the periodic components in the time series.

The 365-day annual cycle is visible in the original time series, but the 30-day and 60-day components are harder to separate by eye.

The spectrum provides a more objective way to identify all three periods.

## 14. Basic statistical concepts for spectra

Before applying Fourier analysis to real climate data, we need to understand several basic statistical concepts.

These concepts control what kinds of variability we can and cannot interpret from the spectrum.

### 14.1 Sampling interval

The sampling interval is the time between two adjacent data points.

For daily data:

$$
\Delta t = 1 \ \text{day}
$$

For monthly data:

$$
\Delta t = 1 \ \text{month}
$$

The same numerical frequency can mean different physical time scales depending on the sampling interval.

In this notebook, we use daily data, so frequency is measured in cycles per day.

### 14.2 Frequency resolution

The smallest frequency spacing is:

$$
\Delta f = \frac{1}{N \Delta t}
$$

where:

- $N$ is the number of time steps.
- $\Delta t$ is the sampling interval.

For a 10-year daily time series:

$$
N \approx 3650
$$

so:

$$
\Delta f \approx \frac{1}{3650} \ \text{cycles per day}
$$

A longer time series gives better frequency resolution.

This means that if we want to study low-frequency variability, we need long records.

### 14.3 Nyquist frequency

The highest frequency that can be resolved is the Nyquist frequency:

$$
f_N = \frac{1}{2\Delta t}
$$

For daily data:

$$
f_N = \frac{1}{2} \ \text{cycles per day}
$$

This corresponds to a period of:

$$
P_N = 2 \ \text{days}
$$

So daily data cannot resolve variability with periods shorter than 2 days.

### 14.4 Record length and low-frequency variability

The longest period we can estimate is limited by the total record length.

For a 10-year synthetic time series, the longest possible period is about 10 years.

However, just because a period can be computed does not mean it can be interpreted with high confidence.

For example, a 10-year record contains:

- many 30-day cycles
- many 60-day cycles
- about 10 annual cycles
- only one 10-year cycle

This means short-period variability is usually estimated more robustly than very long-period variability.

The same idea matters for real data from 1979 to 2024. That record is long enough for subseasonal and interannual analysis, but decadal variability still has relatively few independent cycles.

### 14.5 Power versus amplitude

Amplitude tells us the size of a wave.

Power is proportional to amplitude squared:

$$
Power \propto A^2
$$

If a wave has twice the amplitude, it contributes about four times the power.

This is why power is closely related to variance.

## 15. Key takeaways from Part I

In this part, we learned the basic intuition behind Fourier transform.

The main ideas are:

1. A time series can be viewed in the time domain as $x(t)$.

2. Fourier transform converts the time series into the frequency domain as $X(f)$.

3. Frequency and period are related by:

$$
P = \frac{1}{f}
$$

4. The power spectrum shows how variability is distributed across different periods.

5. Synthetic data are useful because we know the true periods in advance.

6. In our synthetic example, Fourier transform identifies the 30-day, 60-day, and 365-day components.

7. Sampling interval, record length, frequency resolution, and the Nyquist frequency all affect what we can interpret from a spectrum.

In the next part, we will move from Fourier transform to statistical spectral analysis.

That means we will ask whether spectral peaks are reliable, how to smooth noisy spectra, and why red noise matters for climate data.

# Part II. From Fourier transform to statistical spectral analysis

In Part I, we used Fourier transform to convert a time series from the time domain to the frequency domain:

$$
x(t) \rightarrow X(f)
$$

That showed us which frequencies or periods are present in a signal.

In this part, we move from **Fourier transform** to **spectral analysis**.

Fourier transform is the mathematical tool.

Spectral analysis asks the statistical and physical questions:

- Which periods explain most of the variance?
- Are spectral peaks robust or just random noise?
- Why are raw spectra often noisy?
- How can we make the spectrum easier to interpret?
- What kind of background noise should we compare climate spectra against?

These questions matter because real climate data are noisy, autocorrelated, and affected by many physical processes.

## 16. Fourier transform versus spectral analysis

It is useful to separate two related ideas.

### Fourier transform

Fourier transform is a mathematical operation.

It takes a time series:

$$
x(t)
$$

and converts it into frequency space:

$$
X(f)
$$

This tells us how much of each frequency is present in the signal.

### Spectral analysis

Spectral analysis is the process of interpreting the Fourier result.

For example, spectral analysis asks:

- Is there a peak near 30–90 days?
- Does that peak explain a large fraction of variance?
- Is the peak stronger than what we would expect from random noise?
- Does the same time scale appear in OLR, U850, and U200?
- What physical process might explain the peak?

So Fourier transform gives us the frequency ingredients.

Spectral analysis helps us interpret those ingredients statistically and physically.

## 17. Why raw spectra can be noisy

A raw Fourier power spectrum is often called a **periodogram**.

A periodogram is useful because it shows power as a function of frequency or period.

However, raw periodograms are usually noisy.

This means that even if the original time series is random noise, the periodogram can still show many sharp peaks.

This is important:

> A spectral peak does not automatically mean that we have found a real physical oscillation.

In climate science, we usually need additional statistical tools to interpret spectra carefully.

Two common tools are:

1. **Smoothing or averaging the spectrum**
2. **Comparing the spectrum with a red-noise background**

## 18. Compute a raw periodogram

We first compute a raw periodogram for the synthetic signal.

The raw periodogram is similar in spirit to the Fourier power spectrum from Part I.

Here we use `scipy.signal.periodogram`.

The output is a power spectral density estimate.

The frequency unit is cycles per day because our data are daily.

In [ ]:
def compute_raw_periodogram(x, dt=1.0):
    """
    Compute a raw periodogram using scipy.signal.periodogram.

    Parameters
    ----------
    x : array-like
        One-dimensional time series.
    dt : float
        Sampling interval. For daily data, dt = 1 day.

    Returns
    -------
    spec : dict
        Dictionary containing frequency, period, and power spectral density.
    """

    # Convert input to NumPy array
    x = np.asarray(x)

    # Check for missing values
    if np.any(np.isnan(x)):
        raise ValueError(
            "Input time series contains NaN values. "
            "Please remove or fill missing values first."
        )

    # Remove the time mean
    x_anom = x - np.mean(x)

    # Sampling frequency
    # If dt = 1 day, fs = 1 sample per day
    fs = 1.0 / dt

    # Compute periodogram
    freq, psd = signal.periodogram(
        x_anom,
        fs=fs,
        window="boxcar",
        detrend=False,
        scaling="density"
    )

    # Remove zero frequency
    freq = freq[1:]
    psd = psd[1:]

    # Convert frequency to period
    period = 1 / freq

    spec = {
        "frequency": freq,
        "period": period,
        "psd": psd,
    }

    return spec

In [ ]:
raw_spec = compute_raw_periodogram(x, dt=dt)

raw_period = raw_spec["period"]
raw_psd = raw_spec["psd"]

Now we plot the raw periodogram.

This spectrum should show enhanced power near the known periods:

- 365 days
- 60 days
- 30 days

However, we may also see many smaller noisy peaks.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(raw_period, raw_psd, linewidth=1)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel("Period [days]")
ax.set_ylabel("Power spectral density")
ax.set_title("Raw periodogram of the synthetic signal")

ax.set_xlim(2, 1000)
ax.invert_xaxis()

for p in [30, 60, 365]:
    ax.axvline(p, linestyle="--", linewidth=1)
    ax.text(
        p,
        ax.get_ylim()[1] * 0.5,
        f"{p} d",
        rotation=90,
        va="center"
    )

plt.show()

The raw periodogram identifies the major known periods, but it can also look noisy.

This noisiness is not a plotting problem.

It is a statistical property of the raw periodogram.

A raw periodogram has high variance, meaning that individual spectral estimates can fluctuate strongly from one frequency to another.

To make the spectrum easier to interpret, we often smooth or average the spectrum.

## 19. Spectrum smoothing with Welch method

One common way to obtain a smoother spectrum is the **Welch method**.

The Welch method works like this:

1. Divide the time series into shorter overlapping segments.
2. Apply a window function to each segment.
3. Compute a periodogram for each segment.
4. Average the periodograms.

The main advantage is that the resulting spectrum is smoother and more stable.

The main disadvantage is that frequency resolution becomes lower.

This is a common tradeoff in spectral analysis:

> A smoother spectrum usually has lower frequency resolution.

We now write a helper function for the Welch spectrum.

The parameter `nperseg` controls the length of each segment.

A larger `nperseg` gives better frequency resolution but fewer segments to average.

A smaller `nperseg` gives more averaging but poorer frequency resolution.

For the synthetic 10-year daily signal, we use:

$$
nperseg = 1024
$$

This is long enough to include multiple 30-day and 60-day cycles, and also long enough to detect the annual cycle.

In [ ]:
def compute_welch_spectrum(x, dt=1.0, nperseg=1024):
    """
    Compute a smoothed power spectrum using Welch's method.

    Parameters
    ----------
    x : array-like
        One-dimensional time series.
    dt : float
        Sampling interval. For daily data, dt = 1 day.
    nperseg : int
        Length of each segment used in Welch's method.

    Returns
    -------
    spec : dict
        Dictionary containing frequency, period, and power spectral density.
    """

    # Convert input to NumPy array
    x = np.asarray(x)

    # Check for missing values
    if np.any(np.isnan(x)):
        raise ValueError(
            "Input time series contains NaN values. "
            "Please remove or fill missing values first."
        )

    # Remove the time mean
    x_anom = x - np.mean(x)

    # Sampling frequency
    fs = 1.0 / dt

    # Make sure nperseg is not longer than the time series
    nperseg = min(nperseg, len(x_anom))

    # Compute Welch spectrum
    freq, psd = signal.welch(
        x_anom,
        fs=fs,
        window="hann",
        nperseg=nperseg,
        noverlap=nperseg // 2,
        detrend="constant",
        scaling="density"
    )

    # Remove zero frequency
    freq = freq[1:]
    psd = psd[1:]

    # Convert frequency to period
    period = 1 / freq

    spec = {
        "frequency": freq,
        "period": period,
        "psd": psd,
        "nperseg": nperseg,
    }

    return spec

In [ ]:
welch_spec = compute_welch_spectrum(x, dt=dt, nperseg=1024)

welch_period = welch_spec["period"]
welch_psd = welch_spec["psd"]

Now we compare the raw periodogram with the Welch spectrum.

The raw periodogram usually has sharper and noisier peaks.

The Welch spectrum is smoother because it averages spectra from multiple segments.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(raw_period, raw_psd, linewidth=0.8, label="Raw periodogram")
ax.plot(welch_period, welch_psd, linewidth=2, label="Welch spectrum")

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel("Period [days]")
ax.set_ylabel("Power spectral density")
ax.set_title("Raw periodogram versus Welch spectrum")

ax.set_xlim(2, 1000)
ax.invert_xaxis()

for p in [30, 60, 365]:
    ax.axvline(p, linestyle="--", linewidth=1)

ax.legend()

plt.show()

The Welch spectrum is smoother than the raw periodogram.

This makes the broad structure of the spectrum easier to see.

However, smoothing also has a cost.

Because Welch's method averages over shorter segments, it may reduce our ability to distinguish nearby frequencies.

In practice, the choice of `nperseg` depends on the scientific question.

For MJO analysis, we need segments long enough to resolve the 30–90 day band.

## 20. Autocorrelation and red noise

Real climate time series usually have memory.

This means that today's value is often related to yesterday's value.

This is called **autocorrelation**.

The lag-1 autocorrelation is:

$$
r_1 = corr(x_t, x_{t+1})
$$

If $r_1$ is positive, then positive anomalies tend to be followed by positive anomalies, and negative anomalies tend to be followed by negative anomalies.

A simple statistical model for this kind of persistence is an AR(1) model:

$$
x_t = \alpha x_{t-1} + \epsilon_t
$$

where:

- $\alpha$ is the lag-1 autocorrelation parameter
- $\epsilon_t$ is random noise

This model is often called **red noise**.

Red noise tends to have more power at low frequencies than at high frequencies.

This matters for climate spectra because low-frequency peaks can appear even in a simple persistent random process.

We first write a function to calculate lag-1 autocorrelation.

In [ ]:
def lag1_autocorrelation(x):
    """
    Compute lag-1 autocorrelation of a one-dimensional time series.

    Parameters
    ----------
    x : array-like
        One-dimensional time series.

    Returns
    -------
    r1 : float
        Lag-1 autocorrelation.
    """

    x = np.asarray(x)

    # Remove missing values
    x = x[np.isfinite(x)]

    # Remove the mean
    x = x - np.mean(x)

    # Lagged versions
    x0 = x[:-1]
    x1 = x[1:]

    # Correlation between x_t and x_{t+1}
    r1 = np.corrcoef(x0, x1)[0, 1]

    return r1

In [ ]:
r1_x = lag1_autocorrelation(x)

print(f"Lag-1 autocorrelation of the synthetic signal: {r1_x:.3f}")

The lag-1 autocorrelation tells us how persistent the signal is from one day to the next.

For real climate variables, lag-1 autocorrelation is often positive.

This means that a red-noise background is usually more appropriate than a white-noise background.

## 21. Monte Carlo red-noise confidence levels

Now we introduce a simple way to estimate red-noise confidence levels.

The idea is:

1. Estimate the lag-1 autocorrelation of the observed time series.
2. Generate many artificial AR(1) red-noise time series with the same lag-1 autocorrelation and variance.
3. Compute the spectrum of each red-noise time series.
4. At each frequency, calculate the 95th percentile of the red-noise spectra.
5. Compare the observed spectrum with this 95% red-noise level.

If the observed spectrum is above the 95% red-noise level at a given period, then that power is stronger than expected from this simple red-noise model.

This does not prove a physical mechanism, but it provides a useful statistical reference.

First, we write a function to generate AR(1) red noise.

The AR(1) model is:

$$
x_t = \alpha x_{t-1} + \epsilon_t
$$

To make the simulated red noise have approximately the same variance as the original time series, the innovation noise has standard deviation:

$$
\sigma_{\epsilon}
=
\sigma_x \sqrt{1 - \alpha^2}
$$

where $\sigma_x$ is the standard deviation of the original time series.

In [ ]:
def generate_ar1_red_noise(alpha, variance, n, random_state=None):
    """
    Generate one AR(1) red-noise time series.

    Parameters
    ----------
    alpha : float
        AR(1) coefficient. Usually estimated from lag-1 autocorrelation.
    variance : float
        Target variance of the AR(1) process.
    n : int
        Length of the time series.
    random_state : np.random.Generator or None
        Random number generator.

    Returns
    -------
    x_red : ndarray
        Simulated AR(1) red-noise time series.
    """

    if random_state is None:
        random_state = np.random.default_rng()

    # Standard deviation of the full process
    sigma_x = np.sqrt(variance)

    # Standard deviation of the innovation noise
    sigma_epsilon = sigma_x * np.sqrt(1 - alpha**2)

    # Allocate output
    x_red = np.zeros(n)

    # Initialize from the stationary distribution
    x_red[0] = random_state.normal(loc=0.0, scale=sigma_x)

    # Generate AR(1) process
    for i in range(1, n):
        epsilon = random_state.normal(loc=0.0, scale=sigma_epsilon)
        x_red[i] = alpha * x_red[i - 1] + epsilon

    return x_red

Next, we write a function that uses Monte Carlo simulations to estimate the red-noise background and confidence level.

For speed, we use a moderate number of simulations.

For a final scientific analysis, we may want to increase `n_sim`.

In [ ]:
def red_noise_confidence_level(
    x,
    dt=1.0,
    nperseg=1024,
    n_sim=300,
    confidence=95,
    seed=42
):
    """
    Estimate red-noise background and confidence level using Monte Carlo simulation.

    Parameters
    ----------
    x : array-like
        One-dimensional observed time series.
    dt : float
        Sampling interval.
    nperseg : int
        Segment length for Welch spectra.
    n_sim : int
        Number of Monte Carlo AR(1) simulations.
    confidence : float
        Confidence level percentile, for example 95.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    result : dict
        Dictionary containing period, observed spectrum, red-noise mean spectrum,
        confidence spectrum, lag-1 autocorrelation, and Monte Carlo spectra.
    """

    # Convert input to NumPy array and remove missing values
    x = np.asarray(x)
    x = x[np.isfinite(x)]

    # Remove time mean
    x_anom = x - np.mean(x)

    # Basic properties of observed data
    n = len(x_anom)
    variance = np.var(x_anom)
    alpha = lag1_autocorrelation(x_anom)

    # Observed Welch spectrum
    obs_spec = compute_welch_spectrum(
        x_anom,
        dt=dt,
        nperseg=nperseg
    )

    obs_period = obs_spec["period"]
    obs_psd = obs_spec["psd"]

    # Random number generator
    rng = np.random.default_rng(seed)

    # Storage for red-noise spectra
    red_psd_all = []

    for i in range(n_sim):
        # Generate AR(1) red noise
        x_red = generate_ar1_red_noise(
            alpha=alpha,
            variance=variance,
            n=n,
            random_state=rng
        )

        # Compute Welch spectrum using the same settings
        red_spec = compute_welch_spectrum(
            x_red,
            dt=dt,
            nperseg=nperseg
        )

        red_psd_all.append(red_spec["psd"])

    red_psd_all = np.array(red_psd_all)

    # Mean red-noise spectrum
    red_psd_mean = np.mean(red_psd_all, axis=0)

    # Confidence level
    red_psd_conf = np.percentile(red_psd_all, confidence, axis=0)

    result = {
        "period": obs_period,
        "obs_psd": obs_psd,
        "red_psd_mean": red_psd_mean,
        "red_psd_conf": red_psd_conf,
        "alpha": alpha,
        "red_psd_all": red_psd_all,
        "confidence": confidence,
    }

    return result

Now we apply the red-noise confidence calculation to the synthetic signal.

This is mainly for teaching.

Later, we will apply the same logic to real OLR anomalies.

Now we apply the red-noise confidence calculation to the synthetic signal.

This is mainly for teaching.

Later, we will apply the same logic to real OLR anomalies.

In [ ]:
red_result = red_noise_confidence_level(
    x,
    dt=dt,
    nperseg=1024,
    n_sim=300,
    confidence=95,
    seed=42
)

print(f"Lag-1 autocorrelation used for red noise: {red_result['alpha']:.3f}")

Now we plot the observed Welch spectrum together with:

- the mean red-noise spectrum
- the 95% red-noise confidence level

Periods where the observed spectrum exceeds the 95% red-noise level have stronger power than expected from this simple AR(1) red-noise process.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    red_result["period"],
    red_result["obs_psd"],
    linewidth=2,
    label="Observed Welch spectrum"
)

ax.plot(
    red_result["period"],
    red_result["red_psd_mean"],
    linewidth=2,
    linestyle="--",
    label="Mean red-noise spectrum"
)

ax.plot(
    red_result["period"],
    red_result["red_psd_conf"],
    linewidth=2,
    linestyle=":",
    label="95% red-noise level"
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel("Period [days]")
ax.set_ylabel("Power spectral density")
ax.set_title("Spectrum compared with red-noise confidence level")

ax.set_xlim(2, 1000)
ax.invert_xaxis()

for p in [30, 60, 365]:
    ax.axvline(p, linestyle="--", linewidth=1)

ax.legend()

plt.show()

In this synthetic example, we intentionally inserted 30-day, 60-day, and 365-day oscillations.

Therefore, we expect the observed spectrum to show enhanced power near those periods.

The red-noise confidence level helps us ask a more statistical question:

> Are these peaks stronger than what we might expect from a persistent random time series?

For real climate data, this question is important because climate variables often have memory.

Red noise can produce strong low-frequency power even without a true periodic oscillation.

## 22. Important limitations of red-noise testing

Red-noise testing is useful, but it is not perfect.

### 1. AR(1) is a simple model

The AR(1) model assumes that the time series can be described by one lag-1 memory parameter.

Real climate variability is more complicated.

### 2. Significance depends on preprocessing

Removing the seasonal cycle, detrending, or filtering can change the spectrum and the estimated autocorrelation.

### 3. Exceeding the red-noise level does not prove a mechanism

A significant spectral peak tells us that the power is stronger than expected from a simple red-noise null model.

It does not automatically tell us what physical process caused the peak.

### 4. Not exceeding the red-noise level does not prove there is no physics

A physical signal may still exist but be weak, intermittent, or hard to detect with this particular method.

Therefore, red-noise significance should be treated as one piece of evidence, not the final answer.

## 23. Key takeaways from Part II

In this part, we moved from Fourier transform to statistical spectral analysis.

The main ideas are:

1. A raw periodogram can be noisy.

2. A spectral peak does not automatically imply a real physical oscillation.

3. Welch's method smooths the spectrum by averaging spectra from multiple segments.

4. Smoothing makes the spectrum more stable, but reduces frequency resolution.

5. Climate time series often have autocorrelation, meaning they have memory.

6. Red noise is a simple model for persistent climate variability.

7. Red-noise confidence levels help us judge whether spectral power is stronger than expected from an AR(1)-type random process.

In the next part, we will apply these ideas to real OLR data from 1979 to 2024.

We will first examine a broad-spectrum view using demeaned OLR, then move toward MJO-focused OLR anomalies.

# Part III. Real OLR spectrum

In Part I, we learned the basic idea of Fourier transform using synthetic data.

In Part II, we moved from Fourier transform to statistical spectral analysis, including Welch spectra and red-noise confidence levels.

Now we apply these ideas to real daily OLR data.

OLR stands for outgoing longwave radiation. In the tropics, OLR is often used as a proxy for deep convection.

Lower OLR usually indicates colder cloud tops and stronger deep convection.

Higher OLR usually indicates suppressed convection or clearer-sky conditions.

In this part, we will first examine a broad-spectrum view of OLR, then move toward an MJO-focused view.

## 24. Inspect the OLR dataset

We already opened the OLR data from the Pythia cloud storage.

Here we inspect the dataset to check its dimensions, coordinates, and variable names.

In [ ]:
# Extract OLR DataArray from the Dataset
olr_da = olr['olr']

olr_da

## 25. Compute a tropical area-mean OLR time series

For the first real-data example, we compute an area-mean OLR time series over the tropical Indo-Pacific region:

$$
10^\circ S - 10^\circ N,\quad 60^\circ E - 160^\circ E
$$

This region is useful for a first look because MJO-related convection is often active over the tropical Indian Ocean and western Pacific.

We use cosine-latitude weighting when computing the spatial mean.

In [ ]:
def area_mean_robust(da, lat_bounds, lon_bounds, lat_name="lat", lon_name="lon"):
    """
    Compute cosine-latitude-weighted area mean.

    This version handles both increasing and decreasing latitude coordinates.

    Parameters
    ----------
    da : xarray.DataArray
        Input data with latitude and longitude dimensions.
    lat_bounds : tuple
        Latitude range, for example (-10, 10).
    lon_bounds : tuple
        Longitude range in 0-360 convention, for example (60, 160).
    lat_name : str
        Name of latitude coordinate.
    lon_name : str
        Name of longitude coordinate.

    Returns
    -------
    da_mean : xarray.DataArray
        Area-mean time series.
    """

    lat1, lat2 = lat_bounds
    lon1, lon2 = lon_bounds

    # Handle latitude order
    lat_values = da[lat_name].values

    if lat_values[0] < lat_values[-1]:
        lat_slice = slice(lat1, lat2)
    else:
        lat_slice = slice(lat2, lat1)

    # Select region
    da_reg = da.sel(
        {
            lat_name: lat_slice,
            lon_name: slice(lon1, lon2),
        }
    )

    # Cosine-latitude weights
    weights = np.cos(np.deg2rad(da_reg[lat_name]))

    # Weighted spatial mean
    da_mean = da_reg.weighted(weights).mean(dim=(lat_name, lon_name))

    return da_mean

In [ ]:
# Compute tropical Indo-Pacific area-mean OLR
olr_area = area_mean_robust(
    olr_da,
    lat_bounds=lat_bounds,
    lon_bounds=lon_bounds,
    lat_name='lat',
    lon_name='lon',
)

# Load the small area-mean time series into memory
olr_area = olr_area.load()

olr_area

Now we plot the daily area-mean OLR time series.

This is the time-domain view of the real OLR data.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

olr_area.plot(ax=ax, linewidth=0.7)

ax.set_title("Daily OLR averaged over 10°S–10°N, 60°E–160°E")
ax.set_ylabel("OLR")
ax.set_xlabel("Time")

plt.show()

This real OLR time series is more complicated than the synthetic example.

It includes many time scales:

- weather-scale variability
- seasonal variability
- subseasonal variability
- interannual variability
- lower-frequency variability
- random noise

The goal of spectral analysis is to examine how the variance is distributed across these different time scales.

## Fill missing values in the area-mean OLR time series

Before applying Fourier-based spectral analysis, the input time series should not contain missing values.

The area-mean OLR time series may contain a few missing days.

For this tutorial, we fill those missing values using linear interpolation in time.

For an isolated missing day, linear interpolation means that the missing value is replaced by a value between the previous valid day and the next valid day.

If the missing day is exactly halfway between two valid daily values, this is equivalent to using the average of the previous and next valid values.

This is reasonable here because the number of missing days is small and we need a continuous daily time series for spectral analysis.

In [ ]:
# Count missing values before interpolation
n_nan_before = int(olr_area.isnull().sum().values)

print(f"Number of missing values before interpolation: {n_nan_before}")

In [ ]:
# Fill missing values by linear interpolation along time
olr_area = olr_area.interpolate_na(
    dim="time",
    method="linear"
)

# Count missing values after interpolation
n_nan_after = int(olr_area.isnull().sum().values)

print(f"Number of missing values after interpolation: {n_nan_after}")

In [ ]:
# Make sure there are no remaining missing values
if n_nan_after != 0:
    raise ValueError(
        "There are still missing values after interpolation. "
        "Check whether missing values occur at the beginning or end of the time series."
    )

## 26. Check the time axis

Fourier-based methods assume that the time series is evenly spaced.

Our data should be daily, but it is good practice to check the time spacing.

In [ ]:
# Convert time coordinate to pandas datetime index
time_index = pd.to_datetime(olr_area['time'].values)

# Compute time differences
time_diff = np.diff(time_index)

# Show unique time intervals
unique_time_diff = np.unique(time_diff)

unique_time_diff[:10]

For daily data, the time interval should be one day.

The OLR time coordinate may be recorded at 12:00 UTC rather than 00:00 UTC, but that is not a problem.

What matters for Fourier analysis is that adjacent time steps are evenly spaced.

If the data have missing days or irregular spacing, Fourier analysis becomes more complicated.

For this tutorial, we assume the data are regularly spaced daily data after the missing OLR values have been interpolated.

## 27. Broad-spectrum view using demeaned OLR

First, we want a broad-spectrum view.

That means we want to see multiple time scales in the OLR data, including:

- annual variability
- subseasonal variability
- interannual variability
- lower-frequency variability

For this purpose, we remove only the time mean:

$$
x'(t) = x(t) - \bar{x}
$$

We do **not** remove the seasonal cycle yet.

This allows the annual cycle to remain in the spectrum.

The function `compute_welch_spectrum()` removes the time mean internally, so we can pass the interpolated OLR area-mean time series directly.

In [ ]:
# Use the interpolated area-mean OLR time series
olr_area_clean = olr_area

# Convert to NumPy array
x_olr = olr_area_clean.values

# Compute Welch spectrum
# nperseg = 4096 days is about 11.2 years.
# This gives a broad view from daily to interannual time scales.
olr_full_spec = compute_welch_spectrum(
    x_olr,
    dt=dt,
    nperseg=4096,
)

olr_full_period = olr_full_spec["period"]
olr_full_psd = olr_full_spec["psd"]

Now we plot the broad-spectrum OLR spectrum.

We show periods from 2 days to about 10 years.

The vertical reference lines mark:

- 30 days
- 60 days
- 90 days
- 365 days
- 2 years
- 7 years

The shaded region marks the approximate MJO time scale:

$$
30 - 90 \ \text{days}
$$

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(olr_full_period, olr_full_psd, linewidth=2)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel("Period [days]")
ax.set_ylabel("Power spectral density")
ax.set_title("Broad-spectrum view of tropical Indo-Pacific OLR")

ax.set_xlim(2, 3650)
ax.invert_xaxis()

# Highlight approximate MJO band
ax.axvspan(30, 90, alpha=0.2, label="30–90 day band")

# Reference periods
for p, label in [
    (30, "30 d"),
    (60, "60 d"),
    (90, "90 d"),
    (365, "1 yr"),
    (365 * 2, "2 yr"),
    (365 * 7, "7 yr"),
]:
    ax.axvline(p, linestyle="--", linewidth=1)
    ax.text(
        p,
        ax.get_ylim()[1] * 0.5,
        label,
        rotation=90,
        va="center"
    )

ax.legend()

plt.show()

This figure is a broad-spectrum view.

Because we only removed the time mean, the seasonal cycle is still present.

This means we may see strong power near the annual period:

$$
P \approx 365 \ \text{days}
$$

We may also see power in the 30–90 day band, which is the time scale often associated with MJO-like subseasonal variability.

However, this spectrum alone does not prove that the 30–90 day variability is MJO.

MJO also has spatial propagation and coupling between convection and circulation.

## 28. Red-noise reference for the broad-spectrum OLR

Next, we compare the OLR spectrum with a red-noise reference.

This gives a first statistical check.

However, we should be cautious here.

Because this broad-spectrum OLR still contains the seasonal cycle, the red-noise model is not a perfect null hypothesis.

This comparison is useful for teaching, but the more relevant red-noise test for MJO will be applied later to seasonal-cycle-removed OLR anomalies.

In [ ]:
# Monte Carlo red-noise confidence level
# This may take a little time.
olr_full_red = red_noise_confidence_level(
    x_olr,
    dt=dt,
    nperseg=4096,
    n_sim=200,
    confidence=95,
    seed=42,
)

print(f"Lag-1 autocorrelation used for broad-spectrum OLR: {olr_full_red['alpha']:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    olr_full_red["period"],
    olr_full_red["obs_psd"],
    linewidth=2,
    label="Observed Welch spectrum"
)

ax.plot(
    olr_full_red["period"],
    olr_full_red["red_psd_mean"],
    linewidth=2,
    linestyle="--",
    label="Mean red-noise spectrum"
)

ax.plot(
    olr_full_red["period"],
    olr_full_red["red_psd_conf"],
    linewidth=2,
    linestyle=":",
    label="95% red-noise level"
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel("Period [days]")
ax.set_ylabel("Power spectral density")
ax.set_title("Broad-spectrum OLR compared with red-noise reference")

ax.set_xlim(2, 3650)
ax.invert_xaxis()

ax.axvspan(30, 90, alpha=0.2, label="30–90 day band")

for p in [30, 60, 90, 365, 365 * 2, 365 * 7]:
    ax.axvline(p, linestyle="--", linewidth=1)

ax.legend()

plt.show()

This plot compares the observed OLR spectrum with a simple AR(1) red-noise reference.

If the observed spectrum exceeds the 95% red-noise level at a period, then that period has stronger power than expected from this simple persistent-noise model.

However, because the seasonal cycle remains in this broad-spectrum calculation, this is not the cleanest test for MJO.

For MJO-focused analysis, we next remove the mean seasonal cycle.

## 29. Remove leap days before computing daily climatology

To focus on MJO-like variability, we remove the mean seasonal cycle.

For daily data, this is often done by subtracting the day-of-year climatology:

$$
x'(t) = x(t) - \overline{x}_{doy}
$$

where $\overline{x}_{doy}$ is the average value for each calendar day.

Leap years add an extra day, February 29.

To keep the example simple, we remove February 29 before computing the daily climatology.

In [ ]:
def remove_leap_days(da):
    """
    Remove February 29 from a daily time series.

    Parameters
    ----------
    da : xarray.DataArray
        Input daily data with a time coordinate.

    Returns
    -------
    da_no_leap : xarray.DataArray
        Data with February 29 removed.
    """

    is_feb29 = (da["time"].dt.month == 2) & (da["time"].dt.day == 29)

    da_no_leap = da.sel(time=~is_feb29)

    return da_no_leap

## 30. Remove the mean seasonal cycle

Now we compute the daily climatology of the area-mean OLR time series.

Then we subtract it from the original time series.

This gives a seasonal-cycle-removed OLR anomaly.

In [ ]:
def remove_daily_climatology(da):
    """
    Remove daily climatology from a daily time series.

    This function first removes February 29, then computes
    the mean seasonal cycle using day-of-year grouping.

    Parameters
    ----------
    da : xarray.DataArray
        Daily time series with a time coordinate.

    Returns
    -------
    anom : xarray.DataArray
        Seasonal-cycle-removed anomaly.
    clim : xarray.DataArray
        Daily climatology.
    """

    # Remove leap days
    da_no_leap = remove_leap_days(da)

    # Compute daily climatology
    clim = da_no_leap.groupby("time.dayofyear").mean("time")

    # Remove daily climatology
    anom = da_no_leap.groupby("time.dayofyear") - clim

    return anom, clim

In [ ]:
# Remove daily climatology from area-mean OLR
olr_area_anom, olr_area_clim = remove_daily_climatology(olr_area_clean)

olr_area_anom

We can plot the daily climatology to see the mean seasonal cycle that was removed.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

olr_area_clim.plot(ax=ax, linewidth=2)

ax.set_xlabel("Day of year")
ax.set_ylabel("OLR")
ax.set_title("Daily climatology of tropical Indo-Pacific OLR")

plt.show()

Now we plot a short segment of the original area-mean OLR and the seasonal-cycle-removed OLR anomaly.

This helps show the difference between the original time series and the anomaly.

In [ ]:
# Select a short period for visualization
plot_start = "1997-01-01"
plot_end = "1999-12-31"

fig, ax = plt.subplots(figsize=(13, 4))

olr_area_clean.sel(time=slice(plot_start, plot_end)).plot(
    ax=ax,
    linewidth=0.8,
    label="Original OLR"
)

olr_area_anom.sel(time=slice(plot_start, plot_end)).plot(
    ax=ax,
    linewidth=0.8,
    label="Seasonal-cycle-removed OLR anomaly"
)

ax.set_title("Original OLR and seasonal-cycle-removed OLR anomaly")
ax.set_ylabel("OLR")
ax.set_xlabel("Time")
ax.legend()

plt.show()

The original OLR time series contains the mean seasonal cycle.

The anomaly time series removes the regular seasonal cycle and keeps day-to-day, subseasonal, interannual, and other non-seasonal variability.

This version is more appropriate for MJO-focused spectral analysis.

## 31. Spectrum of seasonal-cycle-removed OLR anomaly

Now we compute the Welch spectrum of the seasonal-cycle-removed OLR anomaly.

This is more appropriate for examining MJO-like subseasonal variability.

Here we focus on periods from 10 to 200 days.

In [ ]:
# Convert OLR anomaly to NumPy array
x_olr_anom = olr_area_anom.values

# Make sure there are no missing values
if np.any(np.isnan(x_olr_anom)):
    raise ValueError("OLR anomaly still contains NaN values.")

# Compute Welch spectrum
# nperseg = 2048 days gives a useful spectrum for subseasonal-to-interannual variability
olr_anom_spec = compute_welch_spectrum(
    x_olr_anom,
    dt=dt,
    nperseg=2048,
)

olr_anom_period = olr_anom_spec["period"]
olr_anom_psd = olr_anom_spec["psd"]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(olr_anom_period, olr_anom_psd, linewidth=2)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel("Period [days]")
ax.set_ylabel("Power spectral density")
ax.set_title("Spectrum of seasonal-cycle-removed OLR anomaly")

ax.set_xlim(1, 3000)
ax.invert_xaxis()

# Highlight approximate MJO band
ax.axvspan(30, 90, alpha=0.2, label="30–90 day band")

for p, label in [
    (30, "30 d"),
    (60, "60 d"),
    (90, "90 d"),
]:
    ax.axvline(p, linestyle="--", linewidth=1)
    ax.text(
        p,
        ax.get_ylim()[1] * 0.5,
        label,
        rotation=90,
        va="center"
    )

ax.legend()

plt.show()

This spectrum focuses on the subseasonal range.

The 30–90 day band is often associated with MJO-like variability.

However, a peak in this band alone is not enough to prove that the signal is MJO.

At this stage, we can say:

> The area-mean OLR has variability on MJO-like time scales.

Later, we will examine winds and spatial propagation to connect this signal more directly to MJO physics.

## 32. Red-noise confidence level for OLR anomaly spectrum

Now we apply the red-noise confidence test to the seasonal-cycle-removed OLR anomaly.

This is a cleaner test than the broad-spectrum red-noise comparison because the regular seasonal cycle has been removed.

In [ ]:
olr_anom_red = red_noise_confidence_level(
    x_olr_anom,
    dt=dt,
    nperseg=2048,
    n_sim=200,
    confidence=95,
    seed=42,
)

print(f"Lag-1 autocorrelation used for OLR anomaly: {olr_anom_red['alpha']:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    olr_anom_red["period"],
    olr_anom_red["obs_psd"],
    linewidth=2,
    label="Observed OLR anomaly spectrum"
)

ax.plot(
    olr_anom_red["period"],
    olr_anom_red["red_psd_mean"],
    linewidth=2,
    linestyle="--",
    label="Mean red-noise spectrum"
)

ax.plot(
    olr_anom_red["period"],
    olr_anom_red["red_psd_conf"],
    linewidth=2,
    linestyle=":",
    label="95% red-noise level"
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel("Period [days]")
ax.set_ylabel("Power spectral density")
ax.set_title("OLR anomaly spectrum compared with red-noise confidence level")

ax.set_xlim(10, 2000)
ax.invert_xaxis()

ax.axvspan(30, 90, alpha=0.2, label="30–90 day band")

for p in [30, 60, 90]:
    ax.axvline(p, linestyle="--", linewidth=1)

ax.legend()

plt.show()

In the seasonal-cycle-removed OLR anomaly spectrum, the 30–90 day band contains subseasonal variability, but the signal does not appear as a strong, robust peak above the 95% red-noise level.

This does not necessarily mean that MJO variability is absent.

There are several reasons why an area-mean OLR spectrum may show a weak MJO signal.

First, MJO is an eastward-propagating disturbance. Averaging OLR over a broad Indo-Pacific region can partly cancel anomalies at different longitudes and phases.

Second, MJO is intermittent. It is active during some periods and weak or absent during others, so a spectrum averaged over the full 1979–2024 record may dilute strong individual MJO events.

Third, MJO has strong seasonality. Using all months together can weaken the spectral signature compared with a boreal-winter-only analysis.

Fourth, the red-noise reference can be high if the anomaly time series still contains persistent low-frequency variability.

Therefore, this area-mean spectrum should be interpreted as a first diagnostic of subseasonal OLR variability, not as a complete MJO detection method.

To diagnose MJO more directly, we need to examine circulation coupling with U850 and U200 and spatial propagation using longitude-time or wavenumber-frequency analysis.

This plot asks whether OLR anomaly power in the subseasonal range is stronger than expected from a simple red-noise process.

If the observed spectrum exceeds the 95% red-noise level in part of the 30–90 day band, then that power is statistically stronger than this AR(1) red-noise reference.

This still does not prove that the signal is MJO.

It means that the subseasonal power is stronger than expected from a simple persistent-noise model.

To move closer to an MJO interpretation, we need to examine:

- lower-tropospheric circulation using U850
- upper-tropospheric circulation using U200
- spatial propagation of OLR anomalies

## 33. Key takeaways from Part III

In this part, we applied spectral analysis to real daily OLR.

The main ideas are:

1. A broad-spectrum view removes only the time mean.

2. The broad-spectrum view keeps the seasonal cycle, subseasonal variability, interannual variability, and lower-frequency variability.

3. For MJO-focused analysis, we remove the mean seasonal cycle using daily climatology.

4. The seasonal-cycle-removed OLR anomaly is more appropriate for examining 30–90 day variability.

5. In this area-mean OLR spectrum, the 30–90 day band contains subseasonal variability, but it is not a strong or clearly significant peak above the red-noise reference.

6. This weak spectral peak does not rule out MJO variability because MJO is a propagating and intermittent phenomenon.

7. A spectrum of one broad area-mean time series is not enough to identify MJO. We also need circulation information and spatial propagation.

In the next part, we will add U850 and U200 to examine whether the OLR variability is coupled to lower- and upper-tropospheric circulation.

## 34. Optional but useful: NDJFM-only OLR anomaly spectrum

The annual-mean OLR anomaly spectrum does not show a very strong 30–90 day peak.

One possible reason is that MJO activity is seasonal.

MJO-related convection is often stronger during boreal winter and early spring. Therefore, it can be useful to examine only the NDJFM season:

$$
\text{NDJFM} = \text{November, December, January, February, March}
$$

However, we need to be careful.

We should not simply concatenate all NDJFM days from different years and apply FFT directly.

That would create artificial jumps from March of one year to November of the next year.

Instead, we treat each complete NDJFM season as one continuous segment, compute a spectrum for each season, and then average the spectra across seasons.

This gives a seasonal-mean spectrum for NDJFM OLR anomalies.

### 34.1 Select complete NDJFM seasons

We use the seasonal-cycle-removed OLR anomaly from Part III:

$$
OLR'(t) = OLR(t) - \overline{OLR}_{doy}
$$

Because leap days were removed earlier, a complete NDJFM season has:

$$
30 + 31 + 31 + 28 + 31 = 151 \ \text{days}
$$

We label each NDJFM season by the year of January.

For example:

$$
\text{November 1979 to March 1980}
$$

is labeled as season year 1980.

In [ ]:
def select_complete_ndjfm_seasons(da):
    """
    Select complete NDJFM seasons from a daily time series.

    NDJFM means November, December, January, February, and March.

    Each season is labeled by the year of January.
    For example, Nov 1979-Mar 1980 is labeled as season_year = 1980.

    This function assumes that leap days have already been removed.

    Parameters
    ----------
    da : xarray.DataArray
        Daily time series with a 'time' coordinate.

    Returns
    -------
    seasons : list of xarray.DataArray
        List of complete NDJFM seasonal segments.
    complete_years : ndarray
        Season years corresponding to the returned segments.
    """

    # Select NDJFM months
    ndjfm_months = [11, 12, 1, 2, 3]
    is_ndjfm = da["time"].dt.month.isin(ndjfm_months)

    da_ndjfm = da.sel(time=is_ndjfm)

    # Define season year.
    # Nov-Dec are assigned to the following year.
    season_year = xr.where(
        da_ndjfm["time"].dt.month >= 11,
        da_ndjfm["time"].dt.year + 1,
        da_ndjfm["time"].dt.year
    )

    da_ndjfm = da_ndjfm.assign_coords(
        season_year=("time", season_year.values)
    )

    # Count number of days in each NDJFM season
    counts = da_ndjfm.groupby("season_year").count()

    # A complete no-leap NDJFM season has 151 days
    complete_years = counts["season_year"].where(
        counts == 151,
        drop=True
    ).values

    # Extract complete seasons
    seasons = []

    for year in complete_years:
        season = da_ndjfm.where(
            da_ndjfm["season_year"] == year,
            drop=True
        )

        seasons.append(season)

    return seasons, complete_years

In [ ]:
# Select complete NDJFM seasons from OLR anomaly
olr_ndjfm_seasons, ndjfm_years = select_complete_ndjfm_seasons(
    olr_area_anom
)

print(f"Number of complete NDJFM seasons: {len(olr_ndjfm_seasons)}")
print(f"First complete NDJFM season year: {ndjfm_years[0]}")
print(f"Last complete NDJFM season year: {ndjfm_years[-1]}")
print(f"Length of one NDJFM season: {olr_ndjfm_seasons[0].sizes['time']} days")

### 34.2 Compute an average NDJFM spectrum

Now we compute a spectrum for each complete NDJFM season.

Then we average the spectra across seasons.

Because each NDJFM season is only 151 days long, the frequency resolution is limited.

This means the seasonal spectrum can identify broad subseasonal power, but it cannot resolve frequencies as finely as the full 1979–2024 spectrum.

This is a tradeoff:

- the full-record spectrum has better frequency resolution
- the NDJFM spectrum focuses on the season when MJO activity is often stronger

In [ ]:
def compute_seasonal_mean_periodogram(seasons, dt=1.0):
    """
    Compute a mean periodogram across a list of seasonal segments.

    Parameters
    ----------
    seasons : list of xarray.DataArray
        List of seasonal time series segments.
    dt : float
        Sampling interval. For daily data, dt = 1 day.

    Returns
    -------
    spec : dict
        Dictionary containing frequency, period, mean PSD, and all seasonal PSDs.
    """

    fs = 1.0 / dt

    psd_all = []
    freq_ref = None

    for season in seasons:
        x = season.values

        if np.any(np.isnan(x)):
            raise ValueError("A seasonal segment contains NaN values.")

        # Remove the seasonal-segment mean
        x = x - np.mean(x)

        # Compute periodogram for this season
        freq, psd = signal.periodogram(
            x,
            fs=fs,
            window="hann",
            detrend=False,
            scaling="density"
        )

        # Remove zero frequency
        freq = freq[1:]
        psd = psd[1:]

        if freq_ref is None:
            freq_ref = freq
        else:
            if not np.allclose(freq, freq_ref):
                raise ValueError("Frequency arrays are not identical across seasons.")

        psd_all.append(psd)

    psd_all = np.array(psd_all)

    # Average across seasons
    psd_mean = np.mean(psd_all, axis=0)

    # Convert frequency to period
    period = 1 / freq_ref

    spec = {
        "frequency": freq_ref,
        "period": period,
        "psd_mean": psd_mean,
        "psd_all": psd_all,
    }

    return spec

In [ ]:
# Compute NDJFM mean spectrum
olr_ndjfm_spec = compute_seasonal_mean_periodogram(
    olr_ndjfm_seasons,
    dt=dt
)

olr_ndjfm_period = olr_ndjfm_spec["period"]
olr_ndjfm_psd = olr_ndjfm_spec["psd_mean"]

Now we plot the NDJFM-only OLR anomaly spectrum.

The shaded region marks the 30–90 day band.

Because the NDJFM season is only 151 days long, the spectrum is coarser than the full-record spectrum.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    olr_ndjfm_period,
    olr_ndjfm_psd,
    linewidth=2,
    label="NDJFM mean OLR anomaly spectrum"
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel("Period [days]")
ax.set_ylabel("Power spectral density")
ax.set_title("NDJFM-only OLR anomaly spectrum")

ax.set_xlim(1, 200)
ax.invert_xaxis()

# Highlight approximate MJO band
ax.axvspan(30, 90, alpha=0.2, label="30–90 day band")

for p, label in [
    (30, "30 d"),
    (60, "60 d"),
    (90, "90 d"),
]:
    ax.axvline(p, linestyle="--", linewidth=1)
    ax.text(
        p,
        ax.get_ylim()[1] * 0.5,
        label,
        rotation=90,
        va="center"
    )

ax.legend()

plt.show()

This NDJFM-only spectrum is useful for checking whether subseasonal OLR variability is stronger during the season when MJO activity is often more active.

However, this spectrum should be interpreted carefully.

Each NDJFM segment is only about 151 days long, so the frequency resolution is limited.

This means the spectrum can show broad power in the 30–90 day band, but it may not show a sharp or highly resolved MJO peak.

### 34.3 Add a red-noise reference for the NDJFM spectrum

We can also estimate a red-noise reference for the NDJFM seasonal spectrum.

To keep the comparison consistent, the red-noise simulations use the same structure as the observed calculation:

1. Generate synthetic AR(1) segments with the same length as NDJFM seasons.
2. Compute a periodogram for each synthetic season.
3. Average the synthetic spectra across seasons.
4. Repeat this many times.
5. Estimate the mean red-noise spectrum and the 95% red-noise level.

This gives a red-noise reference for the seasonally averaged NDJFM spectrum.

In [ ]:
def mean_lag1_autocorrelation_across_seasons(seasons):
    """
    Compute the mean lag-1 autocorrelation across seasonal segments.

    Parameters
    ----------
    seasons : list of xarray.DataArray
        List of seasonal time series segments.

    Returns
    -------
    alpha_mean : float
        Mean lag-1 autocorrelation across seasons.
    """

    alpha_all = []

    for season in seasons:
        x = season.values

        if np.any(np.isnan(x)):
            raise ValueError("A seasonal segment contains NaN values.")

        alpha = lag1_autocorrelation(x)
        alpha_all.append(alpha)

    alpha_mean = np.mean(alpha_all)

    return alpha_mean

In [ ]:
def ndjfm_red_noise_confidence_level(
    seasons,
    dt=1.0,
    n_sim=300,
    confidence=95,
    seed=42
):
    """
    Estimate red-noise confidence level for a seasonal-mean spectrum.

    The Monte Carlo simulations mimic the same segment structure
    as the observed seasonal spectrum.

    Parameters
    ----------
    seasons : list of xarray.DataArray
        Observed seasonal segments.
    dt : float
        Sampling interval. For daily data, dt = 1 day.
    n_sim : int
        Number of Monte Carlo simulations.
    confidence : float
        Confidence percentile, for example 95.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    result : dict
        Dictionary containing period, observed PSD, red-noise mean PSD,
        red-noise confidence PSD, and AR(1) parameter.
    """

    # Observed seasonal-mean spectrum
    obs_spec = compute_seasonal_mean_periodogram(
        seasons,
        dt=dt
    )

    obs_period = obs_spec["period"]
    obs_psd = obs_spec["psd_mean"]

    # Estimate AR(1) coefficient from seasonal segments
    alpha = mean_lag1_autocorrelation_across_seasons(seasons)

    # Estimate variance from all seasonal values
    x_all = np.concatenate([season.values for season in seasons])
    variance = np.var(x_all)

    # Segment lengths
    segment_lengths = [season.sizes["time"] for season in seasons]

    # Random number generator
    rng = np.random.default_rng(seed)

    red_psd_all = []

    for i in range(n_sim):
        synthetic_seasons = []

        for n in segment_lengths:
            x_red = generate_ar1_red_noise(
                alpha=alpha,
                variance=variance,
                n=n,
                random_state=rng
            )

            # Convert to DataArray only so the same function can be reused
            red_da = xr.DataArray(
                x_red,
                dims=["time"],
                coords={"time": np.arange(n)}
            )

            synthetic_seasons.append(red_da)

        red_spec = compute_seasonal_mean_periodogram(
            synthetic_seasons,
            dt=dt
        )

        red_psd_all.append(red_spec["psd_mean"])

    red_psd_all = np.array(red_psd_all)

    red_psd_mean = np.mean(red_psd_all, axis=0)
    red_psd_conf = np.percentile(red_psd_all, confidence, axis=0)

    result = {
        "period": obs_period,
        "obs_psd": obs_psd,
        "red_psd_mean": red_psd_mean,
        "red_psd_conf": red_psd_conf,
        "alpha": alpha,
        "confidence": confidence,
        "red_psd_all": red_psd_all,
    }

    return result

In [ ]:
# Estimate NDJFM red-noise confidence level
olr_ndjfm_red = ndjfm_red_noise_confidence_level(
    olr_ndjfm_seasons,
    dt=dt,
    n_sim=300,
    confidence=95,
    seed=42
)

print(f"Mean lag-1 autocorrelation across NDJFM seasons: {olr_ndjfm_red['alpha']:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    olr_ndjfm_red["period"],
    olr_ndjfm_red["obs_psd"],
    linewidth=2,
    label="Observed NDJFM OLR anomaly spectrum"
)

ax.plot(
    olr_ndjfm_red["period"],
    olr_ndjfm_red["red_psd_mean"],
    linewidth=2,
    linestyle="--",
    label="Mean red-noise spectrum"
)

ax.plot(
    olr_ndjfm_red["period"],
    olr_ndjfm_red["red_psd_conf"],
    linewidth=2,
    linestyle=":",
    label="95% red-noise level"
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel("Period [days]")
ax.set_ylabel("Power spectral density")
ax.set_title("NDJFM OLR anomaly spectrum compared with red-noise reference")

ax.set_xlim(1, 200)
ax.invert_xaxis()

ax.axvspan(30, 90, alpha=0.2, label="30–90 day band")

for p in [30, 60, 90]:
    ax.axvline(p, linestyle="--", linewidth=1)

ax.legend()

plt.show()

The NDJFM-only spectrum focuses on the season when MJO activity is often stronger.

If the 30–90 day band is more enhanced in this plot than in the all-month OLR anomaly spectrum, that suggests that seasonal averaging over all months may have diluted the MJO-like subseasonal signal.

However, this result still should not be interpreted as complete MJO identification.

A seasonal OLR spectrum only tells us about time scales.

To diagnose MJO more directly, we still need to examine:

- coupling with U850 and U200
- longitude-time propagation
- ideally, eastward-propagating planetary-scale variability

### 34.4 Interpretation

This NDJFM-only analysis is a useful intermediate step.

It tests whether the weak 30–90 day signal in the full-year area-mean OLR spectrum is partly due to seasonal dilution.

There are three possible outcomes.

If NDJFM shows stronger 30–90 day power than the all-month spectrum, then MJO-like variability is more active during boreal winter and early spring.

If NDJFM still does not show strong 30–90 day power, then the weak signal may be mainly due to area averaging, intermittent events, or the limitations of a one-dimensional OLR spectrum.

If NDJFM power exceeds the red-noise reference, then the seasonal subseasonal signal is stronger than expected from a simple AR(1) process.

In all cases, this remains a time-series spectral diagnostic.

The next step is to examine whether the 30–90 day OLR anomalies are coupled to circulation and whether they propagate eastward.

## Updated interpretation of Part III

The broad-spectrum OLR analysis and the seasonal-cycle-removed OLR anomaly analysis both show subseasonal variability in the 30–90 day range.

However, the 30–90 day band does not appear as a strong, robust peak above the 95% red-noise reference.

The NDJFM-only spectrum also does not show a clearly significant 30–90 day peak.

This does not mean that MJO variability is absent. Instead, it shows that a simple area-mean OLR spectrum is a limited diagnostic for MJO.

There are several reasons:

1. MJO is an eastward-propagating disturbance, so averaging over a broad Indo-Pacific box can cancel anomalies at different longitudes and phases.

2. MJO is intermittent, so a spectrum averaged over many years includes both active and inactive periods.

3. A single NDJFM season is only about 151 days long, so it contains fewer than two 90-day cycles and only about two to three 60-day cycles.

4. OLR contains many other sources of tropical convective variability, not only MJO.

Therefore, this part should be interpreted as a first look at subseasonal OLR variability, not as a complete MJO detection method.

To diagnose MJO more directly, the next step is to examine longitude-time structure and eastward propagation of 30–90 day filtered OLR anomalies.

## Updated interpretation of Part III

The broad-spectrum OLR analysis and the seasonal-cycle-removed OLR anomaly analysis both show subseasonal variability in the 30–90 day range.

However, the 30–90 day band does not appear as a strong, robust peak above the 95% red-noise reference.

The NDJFM-only spectrum also does not show a clearly significant 30–90 day peak.

This does not mean that MJO variability is absent. Instead, it shows that a simple area-mean OLR spectrum is a limited diagnostic for MJO.

There are several reasons:

1. MJO is an eastward-propagating disturbance, so averaging over a broad Indo-Pacific box can cancel anomalies at different longitudes and phases.

2. MJO is intermittent, so a spectrum averaged over many years includes both active and inactive periods.

3. A single NDJFM season is only about 151 days long, so it contains fewer than two 90-day cycles and only about two to three 60-day cycles.

4. OLR contains many other sources of tropical convective variability, not only MJO.

Therefore, this part should be interpreted as a first look at subseasonal OLR variability, not as a complete MJO detection method.

To diagnose MJO more directly, the next step is to examine longitude-time structure and eastward propagation of 30–90 day filtered OLR anomalies.

# Part IV. MJO convection-circulation coupling

In Part III, we examined the OLR spectrum.

The area-mean OLR spectrum showed some 30–90 day variability, but the signal was not a strong or clearly significant MJO peak above the red-noise reference.

This is not surprising.

MJO is an eastward-propagating disturbance, so a broad area mean can partly cancel signals at different longitudes and phases.

However, MJO is not only a convective signal. It is a coupled convection-circulation phenomenon.

Therefore, in this part, we add two wind variables:

- U850: zonal wind at 850 hPa
- U200: zonal wind at 200 hPa

U850 represents lower-tropospheric circulation.

U200 represents upper-tropospheric circulation.

Together with OLR, these variables help us examine whether subseasonal convective variability is linked to large-scale atmospheric circulation.

This part is still based on area-mean time series, so it remains a simplified diagnostic.

A more direct MJO diagnostic will come later when we examine longitude-time propagation.

## 35. Inspect U850 and U200 datasets

We already opened the zonal wind data from the Pythia cloud storage.

Here we inspect `u850` and `u200`.

The coordinate names are:

- `time`
- `lat`
- `lon`

So we use these names directly in the code.

In [ ]:
u200

In [ ]:
# Extract U850 and U200 DataArrays
u850_da = u850["u850"]
u200_da = u200["u200"]

u850_da

In [ ]:
# Select analysis period
u850_da = u850_da.sel(time=slice(start_date, end_date))
u200_da = u200_da.sel(time=slice(start_date, end_date))

u200_da

## 36. Compute tropical area-mean U850 and U200

We compute area-mean U850 and U200 over the same tropical Indo-Pacific region used for OLR:

$$
10^\circ S - 10^\circ N,\quad 60^\circ E - 160^\circ E
$$

Using the same region makes it easier to compare OLR, U850, and U200.

However, this area mean is still a simplified diagnostic. Since MJO propagates eastward, averaging over a broad longitude range can weaken the signal.

In [ ]:
# Compute tropical Indo-Pacific area-mean U850
u850_area = area_mean_robust(
    u850_da,
    lat_bounds=lat_bounds,
    lon_bounds=lon_bounds,
)

# Compute tropical Indo-Pacific area-mean U200
u200_area = area_mean_robust(
    u200_da,
    lat_bounds=lat_bounds,
    lon_bounds=lon_bounds,
)

# Load the small area-mean time series into memory
u850_area = u850_area.load()
u200_area = u200_area.load()

In [ ]:
u850_area

In [ ]:
# Count missing values before interpolation
n_nan_u850_before = int(u850_area.isnull().sum().values)
n_nan_u200_before = int(u200_area.isnull().sum().values)

print(f"Number of missing values in U850 before interpolation: {n_nan_u850_before}")
print(f"Number of missing values in U200 before interpolation: {n_nan_u200_before}")

## 37. Plot area-mean OLR, U850, and U200

Now we plot the area-mean time series.

These are original time series.

They include the seasonal cycle, subseasonal variability, interannual variability, and other fluctuations.

Because the variables have different physical units, this plot is only a first visual check.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

olr_area.plot(ax=ax, linewidth=0.7, label="OLR")
u850_area.plot(ax=ax, linewidth=0.7, label="U850")
u200_area.plot(ax=ax, linewidth=0.7, label="U200")

ax.set_title("Area-mean OLR, U850, and U200")
ax.set_xlabel("Time")
ax.set_ylabel("Original units")
ax.legend()

plt.show()

## 38. Remove the mean seasonal cycle from U850 and U200

For MJO-focused analysis, we remove the mean seasonal cycle from U850 and U200.

This gives seasonal-cycle-removed anomalies:

$$
u'_{850}(t) = u_{850}(t) - \overline{u}_{850,doy}
$$

$$
u'_{200}(t) = u_{200}(t) - \overline{u}_{200,doy}
$$

This matches the OLR anomaly calculation from Part III.

Leap days are removed inside the `remove_daily_climatology()` function.

In [ ]:
# Remove daily climatology from U850 and U200
u850_area_anom, u850_area_clim = remove_daily_climatology(u850_area)
u200_area_anom, u200_area_clim = remove_daily_climatology(u200_area)
u850_area_anom

Now we align OLR, U850, and U200 anomalies on the same time coordinate.

Before aligning, we need to fix two small coordinate issues.

First, the wind anomalies may still carry a `level` coordinate, even though we have already selected one pressure level.

Second, the OLR time coordinate may be recorded at 12:00 UTC, while the wind time coordinate may be recorded at 00:00 UTC.

These refer to the same calendar day, but xarray treats them as different datetime values.

Therefore, we remove the unnecessary `level` coordinate and convert all time coordinates to daily dates before alignment.

In [ ]:
def clean_area_mean_anomaly_coordinates(da):
    """
    Clean coordinates before aligning area-mean anomaly time series.

    This function does two things:

    1. Remove the pressure-level coordinate if it remains after selecting U850 or U200.
    2. Convert time stamps to daily dates so that 00:00 and 12:00 on the same day align.

    Parameters
    ----------
    da : xarray.DataArray
        Area-mean anomaly time series.

    Returns
    -------
    da_clean : xarray.DataArray
        Cleaned DataArray with daily time coordinate.
    """

    da_clean = da.copy()

    # Remove extra level coordinate if it exists
    if "level" in da_clean.coords:
        da_clean = da_clean.drop_vars("level")

    # Convert time stamps to daily dates
    da_clean = da_clean.assign_coords(
        time=da_clean["time"].dt.floor("D")
    )

    return da_clean

In [ ]:
# Clean coordinates before alignment
olr_area_anom_clean = clean_area_mean_anomaly_coordinates(olr_area_anom)
u850_area_anom_clean = clean_area_mean_anomaly_coordinates(u850_area_anom)
u200_area_anom_clean = clean_area_mean_anomaly_coordinates(u200_area_anom)

In [ ]:
# Check time coordinates before alignment
print("OLR first time:", olr_area_anom_clean["time"].values[0])
print("U850 first time:", u850_area_anom_clean["time"].values[0])
print("U200 first time:", u200_area_anom_clean["time"].values[0])

print("OLR length:", olr_area_anom_clean.sizes["time"])
print("U850 length:", u850_area_anom_clean.sizes["time"])
print("U200 length:", u200_area_anom_clean.sizes["time"])# Align all anomaly time series on the same daily time coordinate

In [ ]:
olr_anom_aligned, u850_anom_aligned, u200_anom_aligned = xr.align(
    olr_area_anom_clean,
    u850_area_anom_clean,
    u200_area_anom_clean,
    join="inner"
)

olr_anom_aligned

In [ ]:
# Check aligned lengths
print("Aligned OLR length:", olr_anom_aligned.sizes["time"])
print("Aligned U850 length:", u850_anom_aligned.sizes["time"])
print("Aligned U200 length:", u200_anom_aligned.sizes["time"])

## 39. Compare spectra of OLR, U850, and U200 anomalies

Now we compute Welch spectra for the seasonal-cycle-removed anomalies.

This allows us to ask:

> Do OLR, U850, and U200 all show variability on subseasonal time scales?

We focus on the 10–200 day period range.

Because this is still based on area-mean time series, weak or unclear spectral peaks should not be overinterpreted.

MJO is a propagating disturbance, so an area-mean spectrum is only a simplified diagnostic.

In [ ]:
# Convert aligned anomalies to NumPy arrays
x_olr_anom = olr_anom_aligned.values
x_u850_anom = u850_anom_aligned.values
x_u200_anom = u200_anom_aligned.values

# Check for missing values
if np.any(np.isnan(x_olr_anom)):
    raise ValueError("OLR anomaly contains NaN values.")

if np.any(np.isnan(x_u850_anom)):
    raise ValueError("U850 anomaly contains NaN values.")

if np.any(np.isnan(x_u200_anom)):
    raise ValueError("U200 anomaly contains NaN values.")

In [ ]:
# Compute Welch spectra
olr_spec = compute_welch_spectrum(
    x_olr_anom,
    dt=dt,
    nperseg=2048,
)

u850_spec = compute_welch_spectrum(
    x_u850_anom,
    dt=dt,
    nperseg=2048,
)

u200_spec = compute_welch_spectrum(
    x_u200_anom,
    dt=dt,
    nperseg=2048,
)

Because OLR and wind have different units, their raw power spectral densities are not directly comparable in magnitude.

To compare spectral shapes, we normalize each spectrum by its total spectral power.

This is only for visualization.

It does not change the period where peaks occur.

In [ ]:
def normalize_spectrum_for_plot(psd):
    """
    Normalize a spectrum by its total power for plotting.

    Parameters
    ----------
    psd : array-like
        Power spectral density.

    Returns
    -------
    psd_norm : ndarray
        Normalized spectrum.
    """

    psd = np.asarray(psd)

    psd_norm = psd / np.nansum(psd)

    return psd_norm

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    olr_spec["period"],
    normalize_spectrum_for_plot(olr_spec["psd"]),
    linewidth=2,
    label="OLR"
)

ax.plot(
    u850_spec["period"],
    normalize_spectrum_for_plot(u850_spec["psd"]),
    linewidth=2,
    label="U850"
)

ax.plot(
    u200_spec["period"],
    normalize_spectrum_for_plot(u200_spec["psd"]),
    linewidth=2,
    label="U200"
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel("Period [days]")
ax.set_ylabel("Normalized spectral power")
ax.set_title("Normalized spectra of OLR, U850, and U200 anomalies")

ax.set_xlim(1, 2000)
ax.invert_xaxis()

ax.axvspan(30, 90, alpha=0.2, label="30–90 day band")

for p in [30, 60, 90]:
    ax.axvline(p, linestyle="--", linewidth=1)

ax.legend()

plt.show()

This plot compares the spectral shapes of OLR, U850, and U200 anomalies.

If all three variables show enhanced power in the 30–90 day band, that suggests that subseasonal variability is not limited to convection alone.

However, weak or broad spectral power should not be interpreted as strong MJO evidence.

A spectrum of area-mean variables is a useful first check, but MJO diagnosis also requires spatial propagation and circulation structure.

The 30–90 day band is not visually dominant in this area-mean spectrum.

This does not necessarily mean that MJO-related variability is absent.

It means that the area-mean spectrum is not showing a strong isolated spectral peak.

Because MJO is a propagating phenomenon, averaging over a broad longitude range can reduce the spectral signal.

To make the comparison more quantitative, we can compute the fraction of spectral power that lies within the 30–90 day band.

### 39.1 Fraction of spectral power in the 30–90 day band

We now compute the fraction of spectral power contained in the 30–90 day band.

For each variable, we calculate:

$$
\text{Band power fraction}
=
\frac{
\sum_{30 \leq P \leq 90} PSD(P)
}{
\sum_{10 \leq P \leq 200} PSD(P)
}
$$

Here, the denominator uses the 10–200 day range because we are mainly interested in subseasonal variability.

This diagnostic does not prove that the variability is MJO.

It only tells us how much of the subseasonal spectral power lies in the MJO-like period band.

In [ ]:
def band_power_fraction(period, psd, band=(30, 90), reference_range=(10, 200)):
    """
    Compute the fraction of spectral power within a target period band.

    Parameters
    ----------
    period : array-like
        Period array in days.
    psd : array-like
        Power spectral density.
    band : tuple
        Target period band, for example (30, 90).
    reference_range : tuple
        Period range used as denominator, for example (10, 200).

    Returns
    -------
    frac : float
        Fraction of spectral power in the target band.
    """

    period = np.asarray(period)
    psd = np.asarray(psd)

    band_min, band_max = band
    ref_min, ref_max = reference_range

    # Select target band
    band_mask = (period >= band_min) & (period <= band_max)

    # Select reference range
    ref_mask = (period >= ref_min) & (period <= ref_max)

    # Compute power fraction
    frac = np.nansum(psd[band_mask]) / np.nansum(psd[ref_mask])

    return frac

In [ ]:
# Compute 30-90 day power fraction relative to 10-200 day power
olr_band_frac = band_power_fraction(
    olr_spec["period"],
    olr_spec["psd"],
    band=(30, 90),
    reference_range=(10, 200)
)

u850_band_frac = band_power_fraction(
    u850_spec["period"],
    u850_spec["psd"],
    band=(30, 90),
    reference_range=(10, 200)
)

u200_band_frac = band_power_fraction(
    u200_spec["period"],
    u200_spec["psd"],
    band=(30, 90),
    reference_range=(10, 200)
)

print(f"OLR  30-90 day power fraction: {olr_band_frac:.3f}")
print(f"U850 30-90 day power fraction: {u850_band_frac:.3f}")
print(f"U200 30-90 day power fraction: {u200_band_frac:.3f}")

The 30–90 day power fraction is about 0.53–0.59 for OLR, U850, and U200.

This means that more than half of the 10–200 day spectral power lies within the 30–90 day band for all three variables.

Therefore, even though the spectra do not show a sharp isolated peak, the 30–90 day band is an important part of the area-mean subseasonal variability.

The similar fractions across OLR, U850, and U200 also suggest that subseasonal variability is present in both convection and circulation.

However, this still does not fully identify MJO.

A true MJO diagnosis also requires spatial structure and eastward propagation.

## 40. Low-pass, high-pass, and bandpass filters

Before applying the 30–90 day bandpass filter, it is useful to understand three common types of filters.

### Low-pass filter

A low-pass filter keeps low-frequency variability and removes high-frequency variability.

Since low frequency means long period, a low-pass filter keeps slow variations.

For example, a 90-day low-pass filter keeps variability with periods longer than about 90 days.

This can be useful for seasonal-to-interannual variability.

### High-pass filter

A high-pass filter keeps high-frequency variability and removes low-frequency variability.

Since high frequency means short period, a high-pass filter keeps faster variations.

For example, a 30-day high-pass filter keeps variability with periods shorter than about 30 days.

This can be useful for weather-scale or synoptic-scale variability.

### Bandpass filter

A bandpass filter keeps variability within a selected frequency or period range.

For MJO-focused analysis, we often use a 30–90 day bandpass filter.

This keeps subseasonal variability with periods between 30 and 90 days and removes both faster and slower variability.

In this notebook, the bandpass filter is the most important one because the MJO is usually associated with 30–90 day variability.

The three filters can be summarized as:

| Filter type | Keeps | Removes | Example |
|---|---|---|---|
| Low-pass | Long periods | Short periods | Keep periods longer than 90 days |
| High-pass | Short periods | Long periods | Keep periods shorter than 30 days |
| Bandpass | A selected period range | Both shorter and longer periods | Keep 30–90 day variability |

Here we use the OLR anomaly time series as a simple example.

In [ ]:
def butter_lowpass_filter(x, dt=1.0, cutoff_period=90, order=4):
    """
    Apply a Butterworth low-pass filter to a one-dimensional time series.

    This keeps variability with periods longer than the cutoff period.

    Parameters
    ----------
    x : array-like
        One-dimensional time series.
    dt : float
        Sampling interval. For daily data, dt = 1 day.
    cutoff_period : float
        Cutoff period in days. For example, 90 keeps periods longer than 90 days.
    order : int
        Butterworth filter order.

    Returns
    -------
    x_filtered : ndarray
        Low-pass-filtered time series.
    """

    x = np.asarray(x)

    if np.any(np.isnan(x)):
        raise ValueError("Input time series contains NaN values.")

    # Sampling frequency
    fs = 1.0 / dt

    # Convert cutoff period to cutoff frequency
    cutoff_freq = 1.0 / cutoff_period

    # Nyquist frequency
    nyquist = 0.5 * fs

    # Normalize by Nyquist frequency
    cutoff = cutoff_freq / nyquist

    # Design low-pass filter
    b, a = signal.butter(
        order,
        cutoff,
        btype="lowpass"
    )

    # Apply zero-phase filtering
    x_filtered = signal.filtfilt(b, a, x)

    return x_filtered

In [ ]:
def butter_highpass_filter(x, dt=1.0, cutoff_period=30, order=4):
    """
    Apply a Butterworth high-pass filter to a one-dimensional time series.

    This keeps variability with periods shorter than the cutoff period.

    Parameters
    ----------
    x : array-like
        One-dimensional time series.
    dt : float
        Sampling interval. For daily data, dt = 1 day.
    cutoff_period : float
        Cutoff period in days. For example, 30 keeps periods shorter than 30 days.
    order : int
        Butterworth filter order.

    Returns
    -------
    x_filtered : ndarray
        High-pass-filtered time series.
    """

    x = np.asarray(x)

    if np.any(np.isnan(x)):
        raise ValueError("Input time series contains NaN values.")

    # Sampling frequency
    fs = 1.0 / dt

    # Convert cutoff period to cutoff frequency
    cutoff_freq = 1.0 / cutoff_period

    # Nyquist frequency
    nyquist = 0.5 * fs

    # Normalize by Nyquist frequency
    cutoff = cutoff_freq / nyquist

    # Design high-pass filter
    b, a = signal.butter(
        order,
        cutoff,
        btype="highpass"
    )

    # Apply zero-phase filtering
    x_filtered = signal.filtfilt(b, a, x)

    return x_filtered

Now we apply low-pass and high-pass filters to the seasonal-cycle-removed OLR anomaly.

This is only a demonstration.

The main MJO analysis will still use the 30–90 day bandpass filter in the next section.

In [ ]:
# Apply example filters to OLR anomaly
olr_lowpass_90 = butter_lowpass_filter(
    x_olr_anom,
    dt=dt,
    cutoff_period=90,
    order=4,
)

olr_highpass_30 = butter_highpass_filter(
    x_olr_anom,
    dt=dt,
    cutoff_period=30,
    order=4,
)

In [ ]:
# Convert filtered arrays back to DataArrays
olr_lowpass_90_da = xr.DataArray(
    olr_lowpass_90,
    coords={"time": olr_anom_aligned["time"]},
    dims=["time"],
    name="olr_lowpass_90"
)

olr_highpass_30_da = xr.DataArray(
    olr_highpass_30,
    coords={"time": olr_anom_aligned["time"]},
    dims=["time"],
    name="olr_highpass_30"
)

We plot a short period to compare the original OLR anomaly, the 90-day low-pass component, and the 30-day high-pass component.

The low-pass curve shows slower variability.

The high-pass curve shows faster variability.

In [ ]:
# Choose a shorter period for plotting
plot_start = "1997-01-01"
plot_end = "1999-12-31"

fig, ax = plt.subplots(figsize=(13, 4))

olr_anom_aligned.sel(time=slice(plot_start, plot_end)).plot(
    ax=ax,
    linewidth=0.8,
    label="OLR anomaly"
)

olr_lowpass_90_da.sel(time=slice(plot_start, plot_end)).plot(
    ax=ax,
    linewidth=1.5,
    label="90-day low-pass"
)

olr_highpass_30_da.sel(time=slice(plot_start, plot_end)).plot(
    ax=ax,
    linewidth=0.8,
    label="30-day high-pass"
)

ax.axhline(0, linewidth=0.8)

ax.set_title("Example low-pass and high-pass filtered OLR anomalies")
ax.set_xlabel("Time")
ax.set_ylabel("OLR anomaly")
ax.legend()

plt.show()

This example shows how filters separate different time scales.

The 90-day low-pass component keeps slower variability.

The 30-day high-pass component keeps faster variability.

For MJO analysis, we usually do not want either of these alone.

Instead, we want to isolate the middle range:

$$
30 \leq P \leq 90 \ \text{days}
$$

That is why the next section uses a 30–90 day bandpass filter.

## 41. Bandpass filtering: isolate 30–90 day variability

Spectra tell us which periods are present.

Filtering lets us isolate variability within a chosen period band.

For MJO-focused analysis, a common choice is the 30–90 day band.

A 30–90 day bandpass filter keeps variability with periods between 30 and 90 days and removes variability outside that range.

Here we use a Butterworth bandpass filter.

For daily data, frequency is measured in cycles per day.

The frequency range corresponding to 30–90 days is:

$$
f_{low} = \frac{1}{90}
$$

$$
f_{high} = \frac{1}{30}
$$

The lower frequency corresponds to the longer period.

The higher frequency corresponds to the shorter period.

In [ ]:
def butter_bandpass_filter(x, dt=1.0, low_period=90, high_period=30, order=4):
    """
    Apply a Butterworth bandpass filter to a one-dimensional time series.

    Parameters
    ----------
    x : array-like
        One-dimensional time series.
    dt : float
        Sampling interval. For daily data, dt = 1 day.
    low_period : float
        Long-period cutoff in days. For MJO, use 90 days.
    high_period : float
        Short-period cutoff in days. For MJO, use 30 days.
    order : int
        Butterworth filter order.

    Returns
    -------
    x_filtered : ndarray
        Bandpass-filtered time series.
    """

    x = np.asarray(x)

    if np.any(np.isnan(x)):
        raise ValueError("Input time series contains NaN values.")

    # Sampling frequency
    fs = 1.0 / dt

    # Convert periods to frequencies
    low_freq = 1.0 / low_period
    high_freq = 1.0 / high_period

    # Nyquist frequency
    nyquist = 0.5 * fs

    # Normalize frequencies by Nyquist frequency
    low = low_freq / nyquist
    high = high_freq / nyquist

    # Design Butterworth bandpass filter
    b, a = signal.butter(
        order,
        [low, high],
        btype="bandpass"
    )

    # Apply zero-phase filtering
    x_filtered = signal.filtfilt(b, a, x)

    return x_filtered

Now we apply the 30–90 day bandpass filter to OLR, U850, and U200 anomalies.

We use the aligned seasonal-cycle-removed anomalies.

In [ ]:
# Apply 30-90 day bandpass filter
olr_30_90 = butter_bandpass_filter(
    x_olr_anom,
    dt=dt,
    low_period=90,
    high_period=30,
    order=4,
)

u850_30_90 = butter_bandpass_filter(
    x_u850_anom,
    dt=dt,
    low_period=90,
    high_period=30,
    order=4,
)

u200_30_90 = butter_bandpass_filter(
    x_u200_anom,
    dt=dt,
    low_period=90,
    high_period=30,
    order=4,
)

We convert the filtered NumPy arrays back to xarray DataArrays.

This keeps the time coordinate attached to each filtered time series.

In [ ]:
# Convert filtered arrays back to DataArrays
olr_30_90_da = xr.DataArray(
    olr_30_90,
    coords={"time": olr_anom_aligned["time"]},
    dims=["time"],
    name="olr_30_90"
)

u850_30_90_da = xr.DataArray(
    u850_30_90,
    coords={"time": u850_anom_aligned["time"]},
    dims=["time"],
    name="u850_30_90"
)

u200_30_90_da = xr.DataArray(
    u200_30_90,
    coords={"time": u200_anom_aligned["time"]},
    dims=["time"],
    name="u200_30_90"
)

## 42. Plot 30–90 day filtered anomalies

Now we plot a shorter time period to see the filtered subseasonal variability.

We standardize each filtered anomaly before plotting:

$$
z(t) = \frac{x(t) - \bar{x}}{\sigma_x}
$$

This makes variables with different units easier to compare on the same axis.

For OLR, remember that negative anomalies usually indicate enhanced tropical convection.

In [ ]:
def standardize(x):
    """
    Standardize a time series by removing its mean and dividing by its standard deviation.

    Parameters
    ----------
    x : array-like
        Input time series.

    Returns
    -------
    z : ndarray
        Standardized time series.
    """

    x = np.asarray(x)

    z = (x - np.mean(x)) / np.std(x)

    return z

In [ ]:
# Choose a shorter period for plotting
plot_start = "1997-01-01"
plot_end = "1999-12-31"

# Standardize filtered anomalies
olr_30_90_std = xr.DataArray(
    standardize(olr_30_90_da.values),
    coords={"time": olr_30_90_da["time"]},
    dims=["time"],
    name="OLR"
)

u850_30_90_std = xr.DataArray(
    standardize(u850_30_90_da.values),
    coords={"time": u850_30_90_da["time"]},
    dims=["time"],
    name="U850"
)

u200_30_90_std = xr.DataArray(
    standardize(u200_30_90_da.values),
    coords={"time": u200_30_90_da["time"]},
    dims=["time"],
    name="U200"
)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

olr_30_90_std.sel(time=slice(plot_start, plot_end)).plot(
    ax=ax,
    linewidth=1,
    label="OLR"
)

u850_30_90_std.sel(time=slice(plot_start, plot_end)).plot(
    ax=ax,
    linewidth=1,
    label="U850"
)

u200_30_90_std.sel(time=slice(plot_start, plot_end)).plot(
    ax=ax,
    linewidth=1,
    label="U200"
)

ax.axhline(0, linewidth=0.8)

ax.set_title("Standardized 30–90 day filtered OLR, U850, and U200")
ax.set_xlabel("Time")
ax.set_ylabel("Standardized anomaly")
ax.legend()

plt.show()

This plot shows the 30–90 day filtered variability in OLR, U850, and U200.

Because the variables are standardized, the plot compares timing rather than physical units.

If peaks and troughs tend to occur in a systematic sequence, that suggests a phase relationship among convection and circulation.

However, visual comparison is subjective.

Next, we calculate lead-lag correlations.

## 43. Lead-lag correlation

Lead-lag correlation measures how two time series are related as one is shifted in time relative to the other.

For example, we can ask:

> Is U850 most strongly correlated with OLR at the same time, or does it lead or lag OLR?

We define lag as follows:

- Positive lag means the second variable lags OLR.
- Negative lag means the second variable leads OLR.

For example, if the maximum correlation between OLR and U850 occurs at lag +10 days, then U850 lags OLR by about 10 days.

If the maximum correlation occurs at lag -10 days, then U850 leads OLR by about 10 days.

In [ ]:
def lead_lag_correlation(x, y, max_lag):
    """
    Compute lead-lag correlation between two time series.

    Lag convention:
    - Positive lag means y lags x.
    - Negative lag means y leads x.

    Parameters
    ----------
    x : array-like
        First time series.
    y : array-like
        Second time series.
    max_lag : int
        Maximum lag in time steps.

    Returns
    -------
    lags : ndarray
        Lags from -max_lag to +max_lag.
    r : ndarray
        Correlation at each lag.
    """

    x = np.asarray(x)
    y = np.asarray(y)

    if len(x) != len(y):
        raise ValueError("x and y must have the same length.")

    if np.any(np.isnan(x)) or np.any(np.isnan(y)):
        raise ValueError("Input time series contains NaN values.")

    # Remove means
    x = x - np.mean(x)
    y = y - np.mean(y)

    lags = np.arange(-max_lag, max_lag + 1)
    r = np.empty(len(lags))

    for i, lag in enumerate(lags):
        if lag < 0:
            # y leads x
            x_lag = x[-lag:]
            y_lag = y[:lag]
        elif lag > 0:
            # y lags x
            x_lag = x[:-lag]
            y_lag = y[lag:]
        else:
            # same time
            x_lag = x
            y_lag = y

        r[i] = np.corrcoef(x_lag, y_lag)[0, 1]

    return lags, r

Now we compute lead-lag correlations between:

- OLR and U850
- OLR and U200

We use 30–90 day filtered anomalies.

In [ ]:
# Maximum lag in days
max_lag = 60

# Lead-lag correlation
lags, r_olr_u850 = lead_lag_correlation(
    olr_30_90_da.values,
    u850_30_90_da.values,
    max_lag=max_lag
)

lags, r_olr_u200 = lead_lag_correlation(
    olr_30_90_da.values,
    u200_30_90_da.values,
    max_lag=max_lag
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(lags, r_olr_u850, linewidth=2, label="OLR vs U850")
ax.plot(lags, r_olr_u200, linewidth=2, label="OLR vs U200")

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linestyle="--", linewidth=1)

ax.set_xlabel("Lag [days]")
ax.set_ylabel("Correlation")
ax.set_title("Lead-lag correlation of 30–90 day filtered anomalies")

ax.legend()

plt.show()

The lead-lag correlation shows how circulation anomalies are phased relative to OLR anomalies.

Because low OLR indicates enhanced convection, the physical interpretation of the sign should be made carefully.

A negative OLR anomaly is usually a convective enhancement.

Therefore, a positive correlation with OLR does not mean the same thing as a positive correlation with convection.

If we want a convection-positive index, we can multiply OLR anomalies by -1.

## 44. Use a convection-positive OLR index

For interpretation, it is often useful to define:

$$
C(t) = -OLR'(t)
$$

where $C(t)$ is a convection-positive index.

With this convention:

- positive $C(t)$ means enhanced convection
- negative $C(t)$ means suppressed convection

We now repeat the lead-lag correlation using the convection-positive OLR index.

In [ ]:
# Define convection-positive index
conv_30_90_da = -1 * olr_30_90_da
conv_30_90_da.name = "convection_30_90"

# Lead-lag correlation using convection-positive OLR index
lags, r_conv_u850 = lead_lag_correlation(
    conv_30_90_da.values,
    u850_30_90_da.values,
    max_lag=max_lag
)

lags, r_conv_u200 = lead_lag_correlation(
    conv_30_90_da.values,
    u200_30_90_da.values,
    max_lag=max_lag
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(lags, r_conv_u850, linewidth=2, label="Convection index vs U850")
ax.plot(lags, r_conv_u200, linewidth=2, label="Convection index vs U200")

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linestyle="--", linewidth=1)

ax.set_xlabel("Lag [days]")
ax.set_ylabel("Correlation")
ax.set_title("Lead-lag correlation using convection-positive OLR index")

ax.legend()

plt.show()

This version is easier to interpret physically.

Positive values of the convection index correspond to enhanced tropical convection.

The lead-lag correlation tells us whether lower- and upper-level zonal winds tend to lead, lag, or occur at the same time as convective anomalies.

Because this is based on a broad area mean, the correlations may be weaker than the true propagating MJO structure.

The result should be interpreted as an area-mean coupling diagnostic, not as a complete MJO identification.

## 45. Estimate effective sample size

Filtered time series are autocorrelated.

This means adjacent days are not statistically independent.

Therefore, the actual number of independent samples is smaller than the total number of daily values.

A simple approximation for effective sample size is:

$$
N_{eff}
=
N
\frac{1-r_{1x}r_{1y}}
{1+r_{1x}r_{1y}}
$$

where:

- $N$ is the original sample size.
- $r_{1x}$ is the lag-1 autocorrelation of the first time series.
- $r_{1y}$ is the lag-1 autocorrelation of the second time series.

This approximation is useful for teaching, but it should be interpreted cautiously for strongly filtered data.

In [ ]:
def effective_sample_size(x, y):
    """
    Estimate effective sample size for correlation between two autocorrelated time series.

    Parameters
    ----------
    x : array-like
        First time series.
    y : array-like
        Second time series.

    Returns
    -------
    neff : float
        Effective sample size.
    """

    x = np.asarray(x)
    y = np.asarray(y)

    if len(x) != len(y):
        raise ValueError("x and y must have the same length.")

    # Remove missing values jointly
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    n = len(x)

    r1x = lag1_autocorrelation(x)
    r1y = lag1_autocorrelation(y)

    neff = n * (1 - r1x * r1y) / (1 + r1x * r1y)

    return neff

Now we estimate effective sample size for the filtered convection-wind relationships.

In [ ]:
neff_conv_u850 = effective_sample_size(
    conv_30_90_da.values,
    u850_30_90_da.values
)

neff_conv_u200 = effective_sample_size(
    conv_30_90_da.values,
    u200_30_90_da.values
)

print(f"Effective sample size for convection index vs U850: {neff_conv_u850:.1f}")
print(f"Effective sample size for convection index vs U200: {neff_conv_u200:.1f}")

The effective sample size is usually much smaller than the number of daily time steps.

This happens because the 30–90 day filtered data are highly autocorrelated.

When interpreting correlations from filtered climate data, it is important not to treat every daily value as an independent sample.

## 46. Approximate significance threshold for correlation

For a simple correlation test, the t-statistic can be written as:

$$
t
=
r
\sqrt{
\frac{N_{eff}-2}{1-r^2}
}
$$

where:

- $r$ is the correlation coefficient.
- $N_{eff}$ is the effective sample size.

We can also compute the approximate correlation magnitude needed to pass a two-sided 95% significance test.

This is only an approximate guide because filtered climate data do not fully satisfy the assumptions of a simple independent-sample t-test.

In [ ]:
def correlation_threshold(neff, alpha=0.05):
    """
    Compute approximate two-sided correlation threshold using effective sample size.

    Parameters
    ----------
    neff : float
        Effective sample size.
    alpha : float
        Significance level. Use 0.05 for a two-sided 95% test.

    Returns
    -------
    rcrit : float
        Approximate critical absolute correlation value.
    """

    dof = neff - 2

    if dof <= 0:
        raise ValueError("Effective degrees of freedom must be greater than 2.")

    # Two-sided t critical value
    tcrit = stats.t.ppf(1 - alpha / 2, dof)

    # Convert t critical value to correlation critical value
    rcrit = np.sqrt(tcrit**2 / (tcrit**2 + dof))

    return rcrit

In [ ]:
rcrit_conv_u850 = correlation_threshold(neff_conv_u850, alpha=0.05)
rcrit_conv_u200 = correlation_threshold(neff_conv_u200, alpha=0.05)

print(f"Approximate |r| threshold for convection index vs U850: {rcrit_conv_u850:.3f}")
print(f"Approximate |r| threshold for convection index vs U200: {rcrit_conv_u200:.3f}")

Now we add the approximate significance thresholds to the lead-lag correlation plot.

This gives a visual reference for interpreting the correlations.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(lags, r_conv_u850, linewidth=2, label="Convection index vs U850")
ax.plot(lags, r_conv_u200, linewidth=2, label="Convection index vs U200")

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linestyle="--", linewidth=1)

# Use the larger threshold as a conservative visual reference
rcrit_plot = max(rcrit_conv_u850, rcrit_conv_u200)

ax.axhline(rcrit_plot, linestyle=":", linewidth=1, label="Approx. 95% threshold")
ax.axhline(-rcrit_plot, linestyle=":", linewidth=1)

ax.set_xlabel("Lag [days]")
ax.set_ylabel("Correlation")
ax.set_title("Lead-lag correlation with approximate significance threshold")

ax.legend()

plt.show()

The dotted horizontal lines show an approximate 95% correlation threshold.

Correlations outside these lines are larger than the approximate threshold.

However, this should be interpreted as a rough guide, not as a final rigorous significance test.

For strongly filtered data, more careful methods may be needed.

## 47. Key takeaways from Part IV

In this part, we added U850 and U200 to the OLR analysis.

The main ideas are:

1. MJO is not only a convective signal; it is a coupled convection-circulation phenomenon.

2. OLR is useful for identifying tropical convective variability.

3. U850 and U200 help describe lower- and upper-tropospheric circulation anomalies.

4. Seasonal-cycle-removed OLR, U850, and U200 anomalies can all be examined in the frequency domain.

5. A 30–90 day bandpass filter isolates MJO-like subseasonal variability.

6. Because low OLR indicates enhanced convection, it is useful to define a convection-positive index as $-OLR'$.

7. Lead-lag correlation helps diagnose whether wind anomalies lead, lag, or occur at the same time as convective anomalies.

8. Filtered time series have fewer independent samples than the number of daily time steps.

9. Effective sample size is important when interpreting correlations.

This part still uses broad area-mean time series, so weak spectra or weak correlations should not be interpreted as evidence that MJO is absent.

Instead, it shows the limitation of area-mean diagnostics.

In the next part, we will preserve longitude structure and examine whether 30–90 day OLR anomalies propagate eastward, which is a key feature of MJO.

# Part V. Longitude-time structure and eastward propagation

In Parts III and IV, we used area-mean time series.

That was useful for introducing spectra, filtering, and convection-circulation coupling.

However, MJO is not only a time-scale signal.

A key feature of MJO is **eastward propagation** across the tropical Indo-Pacific region.

A broad area mean can weaken this signal because different longitudes may be in different phases of the MJO.

In this part, we preserve longitude structure.

Instead of averaging OLR over both latitude and longitude, we only average over latitude:

$$
10^\circ S - 10^\circ N
$$

This gives an OLR field with dimensions:

$$
time \times longitude
$$

Then we remove the seasonal cycle, apply a 30–90 day filter, and plot a longitude-time Hovmöller diagram.

This allows us to ask:

> Do tropical OLR anomalies show eastward propagation on 30–90 day time scales?

## 48. Compute equatorial latitude-mean OLR

First, we average OLR over latitude but keep longitude.

We use the same latitude range as before:

$$
10^\circ S - 10^\circ N
$$

The result is a longitude-time field:

$$
OLR(t, \lambda)
$$

where $\lambda$ is longitude.

This is more useful for MJO propagation than a single area-mean time series.

In [ ]:
def lat_mean_robust(da, lat_bounds):
    """
    Compute cosine-latitude-weighted latitude mean while keeping longitude.

    This version handles both increasing and decreasing latitude coordinates.

    Parameters
    ----------
    da : xarray.DataArray
        Input data with time, lat, and lon dimensions.
    lat_bounds : tuple
        Latitude range, for example (-10, 10).

    Returns
    -------
    da_mean : xarray.DataArray
        Latitude-mean field with dimensions time and lon.
    """

    lat1, lat2 = lat_bounds

    # Check whether latitude is increasing or decreasing
    lat_values = da["lat"].values

    if lat_values[0] < lat_values[-1]:
        lat_slice = slice(lat1, lat2)
    else:
        lat_slice = slice(lat2, lat1)

    # Select latitude band
    da_reg = da.sel(lat=lat_slice)

    # Cosine-latitude weights
    weights = np.cos(np.deg2rad(da_reg["lat"]))

    # Weighted latitude mean
    da_mean = da_reg.weighted(weights).mean(dim="lat")

    return da_mean

In [ ]:
# Compute equatorial latitude-mean OLR
olr_eq = lat_mean_robust(
    olr_da,
    lat_bounds=lat_bounds
)

# Load into memory because this field is much smaller than the full gridded data
olr_eq = olr_eq.load()

olr_eq

The resulting DataArray should have dimensions similar to:

$$
time \times lon
$$

This field keeps longitude information, which allows us to examine propagation.

## 49. Fill missing values in the longitude-time OLR field

Before filtering, the input data should not contain missing values.

We check for missing values in the latitude-mean OLR field and fill them using linear interpolation along time.

This is the same idea used earlier for the area-mean OLR time series, but now it is applied at each longitude.

In [ ]:
# Count missing values before interpolation
n_nan_olr_eq_before = int(olr_eq.isnull().sum().values)

print(f"Number of missing values before interpolation: {n_nan_olr_eq_before}")

In [ ]:
# Fill missing values by linear interpolation along time
olr_eq = olr_eq.interpolate_na(
    dim="time",
    method="linear"
)

# Count missing values after interpolation
n_nan_olr_eq_after = int(olr_eq.isnull().sum().values)

print(f"Number of missing values after interpolation: {n_nan_olr_eq_after}")

In [ ]:
# Make sure there are no remaining missing values
if n_nan_olr_eq_after != 0:
    raise ValueError(
        "There are still missing values after interpolation. "
        "Check whether missing values occur at the beginning or end of the time series."
    )

After interpolation, the longitude-time OLR field should contain no missing values.

This is necessary before applying the bandpass filter.

## 50. Remove the mean seasonal cycle at each longitude

For MJO-focused analysis, we remove the mean seasonal cycle at each longitude.

This gives:

$$
OLR'(t, \lambda) = OLR(t, \lambda) - \overline{OLR}_{doy}(\lambda)
$$

where $\overline{OLR}_{doy}(\lambda)$ is the day-of-year climatology at each longitude.

This removes the regular seasonal cycle but keeps subseasonal anomalies.

In [ ]:
# Remove daily climatology at each longitude
olr_eq_anom, olr_eq_clim = remove_daily_climatology(olr_eq)

# Convert time stamps to daily dates
# This avoids issues caused by 12:00 UTC versus 00:00 UTC time stamps.
olr_eq_anom = olr_eq_anom.assign_coords(
    time=olr_eq_anom["time"].dt.floor("D")
)

olr_eq_anom

The anomaly field still has longitude structure.

This is the key difference from the area-mean analysis.

Instead of one time series, we now have one time series at each longitude.

## 51. Apply the 30–90 day bandpass filter at each longitude

Now we apply the 30–90 day bandpass filter to the OLR anomaly at each longitude.

This keeps MJO-like subseasonal variability and removes both faster and slower variability.

Because the filter is applied separately at each longitude, the output still has dimensions:

$$
time \times lon
$$

In [ ]:
# Make sure the data are ordered as time, lon
olr_eq_anom = olr_eq_anom.transpose("time", "lon")

# Check for missing values
if np.any(np.isnan(olr_eq_anom.values)):
    raise ValueError("Equatorial OLR anomaly contains NaN values.")

In [ ]:
# Allocate output array
olr_eq_30_90_values = np.empty_like(
    olr_eq_anom.values,
    dtype=np.float64
)

# Apply the bandpass filter at each longitude
for j in range(olr_eq_anom.sizes["lon"]):
    x_lon = olr_eq_anom.isel(lon=j).values

    olr_eq_30_90_values[:, j] = butter_bandpass_filter(
        x_lon,
        dt=dt,
        low_period=90,
        high_period=30,
        order=4,
    )

# Convert back to DataArray
olr_eq_30_90_da = xr.DataArray(
    olr_eq_30_90_values,
    coords=olr_eq_anom.coords,
    dims=olr_eq_anom.dims,
    name="olr_eq_30_90"
)

olr_eq_30_90_da

The filtered field contains OLR variability in the 30–90 day period band.

Because negative OLR anomalies usually indicate enhanced convection, it is often easier to define a convection-positive variable:

$$
C(t, \lambda) = -OLR'_{30-90}(t, \lambda)
$$

With this convention:

- positive values indicate enhanced convection
- negative values indicate suppressed convection

In [ ]:
# Define convection-positive 30-90 day filtered OLR
conv_eq_30_90_da = -1 * olr_eq_30_90_da
conv_eq_30_90_da.name = "convection_eq_30_90"

conv_eq_30_90_da

## 52. Plot a longitude-time Hovmöller diagram

A Hovmöller diagram shows how a variable changes with both longitude and time.

Here we plot the 30–90 day filtered convection-positive OLR anomaly.

The x-axis is longitude.

The y-axis is time.

Eastward propagation appears as coherent tilted features that move from smaller longitudes to larger longitudes as time advances.

We first select an example time period.

You can change `hov_start` and `hov_end` to inspect other periods.

In [ ]:
# Example period for the Hovmöller diagram
hov_start = "1997-01-01"
hov_end = "1997-06-30"

# Longitude range for plotting
hov_lon_bounds = (40, 200)

In [ ]:
# Select data for plotting
conv_hov = conv_eq_30_90_da.sel(
    time=slice(hov_start, hov_end),
    lon=slice(hov_lon_bounds[0], hov_lon_bounds[1])
)

conv_hov

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

# Plot Hovmöller diagram
p = conv_hov.plot.contourf(
    ax=ax,
    x="lon",
    y="time",
    levels=np.linspace(-30, 30, 21),
    cmap="RdBu_r",
    extend="both",
    add_colorbar=True,
)

ax.set_title("30–90 day filtered convection-positive OLR anomaly")
ax.set_xlabel("Longitude [degrees east]")
ax.set_ylabel("Time")

# Put earlier dates at the top and later dates at the bottom.
# With this convention, eastward propagation appears as downward-right tilted features.
ax.invert_yaxis()

plt.show()

In this plot, positive values correspond to enhanced convection because we plotted:

$$
-OLR'_{30-90}
$$

A coherent eastward-propagating MJO-like signal would appear as a tilted band moving from west to east with time.

With the y-axis inverted, eastward propagation appears as a downward-right tilt.

If the pattern is noisy or discontinuous, that does not necessarily mean MJO is absent.

MJO events are intermittent, and some periods are more active than others.

## 53. Plot multiple candidate periods

MJO activity is intermittent.

A single selected period may or may not contain a strong MJO event.

To make the notebook more useful, we define a small helper function that allows us to plot different time windows.

This makes it easier to search for periods with clearer eastward propagation.

In [ ]:
def plot_olr_hovmoller(
    da,
    start,
    end,
    lon_bounds=(40, 200),
    vmin=-30,
    vmax=30,
    title=None
):
    """
    Plot a longitude-time Hovmöller diagram.

    Parameters
    ----------
    da : xarray.DataArray
        Longitude-time field with dimensions time and lon.
    start : str
        Start date, for example "1997-01-01".
    end : str
        End date, for example "1997-06-30".
    lon_bounds : tuple
        Longitude range for plotting.
    vmin : float
        Minimum contour value.
    vmax : float
        Maximum contour value.
    title : str or None
        Plot title.
    """

    da_plot = da.sel(
        time=slice(start, end),
        lon=slice(lon_bounds[0], lon_bounds[1])
    )

    levels = np.linspace(vmin, vmax, 21)

    fig, ax = plt.subplots(figsize=(11, 5))

    da_plot.plot.contourf(
        ax=ax,
        x="lon",
        y="time",
        levels=levels,
        cmap="RdBu_r",
        extend="both",
        add_colorbar=True,
    )

    if title is None:
        title = f"30–90 day filtered convection-positive OLR: {start} to {end}"

    ax.set_title(title)
    ax.set_xlabel("Longitude [degrees east]")
    ax.set_ylabel("Time")

    # Earlier dates at top, later dates at bottom
    ax.invert_yaxis()

    plt.show()

In [ ]:
plot_olr_hovmoller(
    conv_eq_30_90_da,
    start="1997-01-01",
    end="1997-06-30",
    lon_bounds=(40, 200),
    vmin=-30,
    vmax=30,
    title="Example Hovmöller diagram: 1997 Jan-Jun"
)

In [ ]:
plot_olr_hovmoller(
    conv_eq_30_90_da,
    start="2004-11-01",
    end="2005-04-30",
    lon_bounds=(40, 200),
    vmin=-30,
    vmax=30,
    title="Example Hovmöller diagram: 2004 Nov-2005 Apr"
)

In [ ]:
plot_olr_hovmoller(
    conv_eq_30_90_da,
    start="2015-11-01",
    end="2016-04-30",
    lon_bounds=(40, 200),
    vmin=-30,
    vmax=30,
    title="Example Hovmöller diagram: 2015 Nov-2016 Apr"
)

These examples may show different degrees of organization.

Some periods may show clear eastward-moving convective anomalies.

Other periods may look weaker or more disorganized.

This is expected because MJO is intermittent.

A Hovmöller diagram is useful because it reveals information that a single area-mean spectrum cannot show.

## 54. Optional: compare unfiltered and filtered longitude-time OLR

It is useful to compare the original anomaly field with the 30–90 day filtered field.

The unfiltered anomaly contains many time scales at once.

The filtered field isolates subseasonal variability.

This comparison helps show what the bandpass filter is doing in longitude-time space.

In [ ]:
# Select the same period and longitude range
olr_unfiltered_hov = olr_eq_anom.sel(
    time=slice(hov_start, hov_end),
    lon=slice(hov_lon_bounds[0], hov_lon_bounds[1])
)

olr_filtered_hov = olr_eq_30_90_da.sel(
    time=slice(hov_start, hov_end),
    lon=slice(hov_lon_bounds[0], hov_lon_bounds[1])
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

olr_unfiltered_hov.plot.contourf(
    ax=ax,
    x="lon",
    y="time",
    levels=np.linspace(-60, 60, 25),
    cmap="RdBu_r",
    extend="both",
    add_colorbar=True,
)

ax.set_title("Unfiltered equatorial OLR anomaly")
ax.set_xlabel("Longitude [degrees east]")
ax.set_ylabel("Time")
ax.invert_yaxis()

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

olr_filtered_hov.plot.contourf(
    ax=ax,
    x="lon",
    y="time",
    levels=np.linspace(-30, 30, 21),
    cmap="RdBu_r",
    extend="both",
    add_colorbar=True,
)

ax.set_title("30–90 day filtered equatorial OLR anomaly")
ax.set_xlabel("Longitude [degrees east]")
ax.set_ylabel("Time")
ax.invert_yaxis()

plt.show()

The unfiltered anomaly contains many time scales, including faster weather-scale variability and slower variability.

The 30–90 day filtered anomaly emphasizes subseasonal variability.

If MJO-like activity is present, the filtered field should make the eastward-propagating structure easier to see.

## 55. Interpretation of the Hovmöller diagram

The Hovmöller diagram provides a more direct view of MJO-like propagation than an area-mean spectrum.

A useful interpretation guide is:

- Coherent tilted bands indicate organized propagation.
- Downward-right tilt indicates eastward propagation when the y-axis is inverted.
- Positive convection-positive OLR anomalies indicate enhanced convection.
- Negative convection-positive OLR anomalies indicate suppressed convection.
- Disorganized patterns suggest weak, intermittent, or noisy subseasonal convection.

This diagnostic still does not fully isolate MJO from all other tropical variability, but it is much closer to the physical definition of MJO than a single area-mean spectrum.

A Hovmöller diagram gives a visual diagnosis of propagation.

A more advanced and quantitative method is wavenumber-frequency analysis, which separates eastward- and westward-propagating variability in longitude-time data.

That method will be introduced in a separate notebook.

## 56. Key takeaways from Part V

In this part, we moved from area-mean time series to longitude-time structure.

The main ideas are:

1. MJO is a propagating phenomenon, so area-mean diagnostics can weaken the signal.

2. Latitude-mean OLR preserves longitude structure while reducing small-scale noise.

3. Removing the daily climatology gives seasonal-cycle-removed OLR anomalies at each longitude.

4. A 30–90 day bandpass filter isolates MJO-like subseasonal variability.

5. A Hovmöller diagram shows how filtered OLR anomalies move with longitude and time.

6. Eastward propagation appears as coherent tilted features moving from smaller longitudes to larger longitudes as time advances.

7. Hovmöller diagrams are more physically informative for MJO than a single area-mean OLR spectrum.

8. MJO is intermittent, so not every selected period will show strong eastward propagation.

The area-mean spectra show that 30–90 day variability is present, but the spectral peak is not as sharp as in classical RMM-based diagnostics.

This difference is expected. Wheeler and Hendon (2004) first projected OLR, U850, and U200 onto multivariate EOFs that capture the large-scale MJO structure. The spectra are then computed from the resulting PC time series, which are already spatially filtered MJO indices.

In contrast, the present notebook uses simple area-mean time series. This is useful for teaching Fourier spectra and filtering, but it is not optimized to isolate MJO.